# various Models for Hyperspectral and Lidar Image Classification



In [ ]:
import sys
sys.path.append("./../")
import matplotlib.pyplot as plt
from torchsummary import summary
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, cohen_kappa_score
import math
from PIL import Image
import time
from scipy.io import loadmat as loadmat
from scipy import io
import random
import numpy as np
import os
import torch 
import torch.utils.data as dataf
import torch.nn as nn
from operator import truediv
import record
# from FocalLoss import FocalLoss
import pandas as pd
import seaborn as sns
# from dataset import Multimodal_Dataset_Train, Multimodal_Dataset_Test
import torch.backends.cudnn as cudnn
cudnn.deterministic = True
cudnn.benchmark = False
from util_torch import *
from torch.utils.data import DataLoader, TensorDataset
from collections import Counter
from torch.autograd import Variable
import sklearn.model_selection
from sklearn import metrics, preprocessing
from collections import deque
import torch.nn.functional as F
from torchvision import models
from thop import profile
import record
import datetime
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# labels = loadmat('./datasets/trento_data.mat')['ground']
# data_hsi_o = np.array(loadmat('./datasets/trento_data.mat')['HSI_data'])
# data_lidar_o = np.array(loadmat('./datasets/trento_data.mat')['LiDAR_data'])
# 设置随机种子
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(4)


labels = loadmat('./datasets/DFC2013_gt.mat')['DFC2013_gt']
data_hsi_o = np.array(loadmat('./datasets/houston_data.mat')['hsi'])
data_lidar_o = np.array(loadmat('./datasets/houston_data.mat')['lidar'])

# datasetName = 'muufl'
# labels = loadmat('./datasets/MUUFL_gt.mat')['labels']
# data_hsi_o = np.array(loadmat('./datasets/MUUFL.mat')['data'])
# data_lidar_o = np.array(loadmat('./datasets/muufl_lidar.mat')['z'])

data_lidar_o = np.reshape(data_lidar_o, (data_lidar_o.shape[0], data_lidar_o.shape[1], 1))

datasetName = 'houston2013'
num_classes = np.max(np.max(labels))
hsi_band = data_hsi_o.shape[2]
lidar_band = data_lidar_o.shape[2]
itm = 1

In [ ]:
patchesData_lidar, patchesLabels = createPatches(data_lidar_o, labels, windowSize=11)
patchesLabels = patchesLabels.astype(np.int32)

# train_sample = (1162, 214, 344, 91, 334, 24, 112, 312, 70, 30, 30)
# validate_sample = (1162, 214, 344, 91, 334, 24, 112, 312, 70, 30, 30)

# train_sample = (150, 150, 150, 150, 150, 50, 150, 150, 150, 50, 50)
# validate_sample = (150, 150, 150, 150, 150, 50, 150, 150, 150, 50, 50)

#houston 5%
# train_sample = (62,63,35, 62, 62, 16, 63, 62, 63, 61, 62, 62, 23, 21, 33)
# validate_sample = (62,63,35, 62, 62, 16, 63, 62, 63, 61, 62, 62, 23, 21, 33)

#houston 3%
train_sample = (38,38,21, 37, 37, 10, 38, 37, 38, 37, 37, 37, 14, 13, 18)
validate_sample = (38,38,21, 37, 37, 10, 38, 37, 38, 37, 37, 37, 14, 13, 18)

# train_sample = (12,12,7, 12, 12, 3, 13, 12, 13, 12, 12, 12, 5, 4, 6)
# validate_sample = (12,12,7, 12, 12, 3, 13, 12, 13, 12, 12, 12, 5, 4, 6)

# train_sample = [i*10 for i in train_sample]
# validate_sample = [i*5 for i in validate_sample]

# train_sample = (50, 50, 50, 50, 50, 50)
# validate_sample = (50, 50, 50, 50, 50, 50)

trainIndex, valIndex, testIndex = random_sample(train_sample, validate_sample, patchesLabels)

train_gt = np.zeros_like(labels).reshape(np.prod(labels.shape[:2]),order = "F")
label_list = labels.reshape(np.prod(labels.shape[:2]),order = "F")
train_gt[trainIndex] = label_list[trainIndex]
train_gt = train_gt.reshape((labels.shape[0],labels.shape[1]),order = "F")

val_gt = np.zeros_like(labels).reshape(np.prod(labels.shape[:2]),order = "F")
val_gt[valIndex] = label_list[valIndex]
val_gt = val_gt.reshape((labels.shape[0],labels.shape[1]),order = "F")

test_gt = np.zeros_like(labels).reshape(np.prod(labels.shape[:2]),order = "F")
test_gt[testIndex] = label_list[testIndex]
test_gt = test_gt.reshape((labels.shape[0],labels.shape[1]),order = "F")

train_size = len(trainIndex)


In [40]:
# gt = labels
# indices = np.nonzero(gt)
# X = list(zip(*indices))
# y = gt[indices] 
# train_gt = np.zeros_like(gt)
# test_gt = np.zeros_like(gt)
# val_gt = np.zeros_like(gt)
# print('train_gt: ',type(train_gt),train_gt.shape)

# train_indices, test_indices = sklearn.model_selection.train_test_split(X, train_size=0.05, random_state=12,stratify=y)

# val_size = len(train_indices)
# val_indices = test_indices[-val_size:]
# test_indices = test_indices[:-val_size]

# train_size = len(train_indices)
# train_indices = list(zip(*train_indices))
# val_indices = list(zip(*val_indices))
# test_indices = list(zip(*test_indices))

# train_gt[train_indices] = gt[train_indices]
# val_gt[val_indices] = gt[val_indices]
# test_gt[test_indices] = gt[test_indices]

In [41]:
trainIndex

array([409605,  83549, 452803, 252228,  92668, 333139, 408584, 271651,
       184310,  74517, 102811, 325208, 257522,  89845,  64275, 612030,
        89462,  59377, 577079,  69164,  63981, 339433, 204437, 268401,
       461325,  95459, 322174, 450931, 246123, 398072, 569248, 258217,
       438188, 444021,  68352, 428903, 408987, 320331, 360490, 439715,
       658880, 344804, 183134, 269450,  97547, 478418, 372239,  98940,
       351996, 341299,  89787, 211949, 350664, 591662, 234731, 165378,
         5111, 211598, 341662, 592958,  66719,  71273, 607671, 437849,
       150977, 428205,  28213,  11371, 439567, 336288, 274465, 606885,
       286818, 149578, 343742, 172364, 100361, 268042, 628824, 225581,
       494506, 253926, 248370, 642939, 439563,   9551,  76616, 258920,
        97554, 281329, 408169, 570472, 122024, 351011, 241559, 409965,
       453397, 333145, 101731,  25387, 393185,  99979, 351865, 515565,
       462370, 263892,  84661,  84869, 409968, 403622, 351012, 522512,
      

In [ ]:
sys.path.append("./mcf/")
from MCF_demo_1 import *
from config import GlobalConfig

config = GlobalConfig()
windowSize = 11
BATCH_SIZE = 16
EPOCH = 300
FileName = 'mcf'

data = data_hsi_o.reshape(np.prod(data_hsi_o.shape[:2]), np.prod(data_hsi_o.shape[2:]))
data = preprocessing.scale(data)
data_hsi = data.reshape(data_hsi_o.shape[0], data_hsi_o.shape[1], data_hsi_o.shape[2])

data_2 = data_lidar_o.reshape(np.prod(data_lidar_o.shape[:2]), np.prod(data_lidar_o.shape[2:]))
data_2 = preprocessing.scale(data_2)
data_lidar = data_2.reshape(data_lidar_o.shape[0], data_lidar_o.shape[1],data_lidar_o.shape[2])


# scaler = preprocessing.MinMaxScaler()
# data = scaler.fit_transform(data)
# data_hsi_o = data.reshape(data_hsi_o.shape[0], data_hsi_o.shape[1], data_hsi_o.shape[2])

# data_hsi, _ = applyPCA(data_hsi_o, numComponents=numComponents)
# data_lidar, _ = applyPCA(data_lidar, numComponents=1)

train_dataset = Multidata(data_hsi, data_lidar, train_gt, windowSize)
train_loader = torch.utils.data.DataLoader(train_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)

val_dataset = Multidata(data_hsi, data_lidar, val_gt, windowSize)
val_loader = torch.utils.data.DataLoader(val_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)

test_dataset = Multidata(data_hsi, data_lidar, test_gt, windowSize)
test_loader = torch.utils.data.DataLoader(test_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)
y_temp = np.ones_like(labels)
all_dataset = Multidata(data_hsi, data_lidar, y_temp, windowSize)
all_map_loader = torch.utils.data.DataLoader(all_dataset,
                               batch_size=256,
                               shuffle=False)



KAPPA = []
OA = []
AA = []
ELEMENT_ACC = np.zeros((itm, num_classes))
training_time = []
testing_time = []
infer_time = []

for itera in range(0, itm):
    
#     print("---------------------------------- Model Summary ---------------------------------------------")
    # model = Encoder(hsi_band, lidar_band, num_classes, config).cuda()
    model = MCF(hsi_band, lidar_band, num_classes).cuda()
    summary(model,[(hsi_band, windowSize,windowSize),(lidar_band, windowSize,windowSize)]) 

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4,weight_decay=5e-3)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.7)
    loss_func = nn.CrossEntropyLoss()
    
    model, train_time = train(model, loss_func, DEVICE, train_loader, optimizer, scheduler, EPOCH, val_loader, itera)
    test_acc_temp, test_loss_temp, y_pred, target, test_time = test(model, DEVICE, test_loader)
    oa,aa,kappa,each_acc, accuracy_matrix = reports(y_pred, target)
    time_infer = infer_allmap(model, DEVICE, datasetName, FileName, labels, all_map_loader, itera)
    
    input_1 = torch.randn(1, hsi_band, windowSize, windowSize)
    input_2 = torch.randn(1, lidar_band, windowSize, windowSize)
    input_1 = input_1.cuda()
    input_2 = input_2.cuda()
    macs, params = profile(model, inputs=(input_1,input_2, ))
    print("params, macs", params, macs)
    
    training_time.append(train_time)
    testing_time.append(test_time)
    infer_time.append(time_infer)
    KAPPA.append(kappa)
    OA.append(oa)
    AA.append(aa)
    ELEMENT_ACC[itera, :] = each_acc
    current_time = datetime.datetime.now()
    record.record_output(OA, AA, KAPPA, ELEMENT_ACC, training_time, testing_time, infer_time, macs, params,train_sample, './' + FileName + '/' +
                         datasetName  + '_' + str(train_size) + '_' + str(params) + str(current_time) + '_Report' +'.txt')

    print("final test results :", accuracy_matrix)

    del model,input_1,input_2


/home/admin123/anaconda3/envs/mcf/lib/python3.8/site-packages/sklearn/preprocessing/_data.py:246: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/home/admin123/anaconda3/envs/mcf/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/admin123/anaconda3/envs/mcf/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Small_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


RuntimeError: mat1 and mat2 shapes cannot be multiplied (256x16 and 25x16)

In [6]:
sys.path.append("./mft/")
from mft_model import *


windowSize = 11
BATCH_SIZE = 32
EPOCH = 200
FM = 16
num_heads = 8
mlp_dim = 512
depth = 2
ntokens = 4
FileName = 'mft'

data = data_hsi_o.reshape(np.prod(data_hsi_o.shape[:2]), np.prod(data_hsi_o.shape[2:]))
scaler = preprocessing.MinMaxScaler()
data = scaler.fit_transform(data)
data_hsi = data.reshape(data_hsi_o.shape[0], data_hsi_o.shape[1], data_hsi_o.shape[2])
data_lidar = data_lidar_o


# data_hsi, _ = applyPCA(data_hsi_o, numComponents=numComponents)
# data_lidar, _ = applyPCA(data_lidar, numComponents=1)

train_dataset = Multidata(data_hsi, data_lidar, train_gt, windowSize)
train_loader = torch.utils.data.DataLoader(train_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)

val_dataset = Multidata(data_hsi, data_lidar, val_gt, windowSize)
val_loader = torch.utils.data.DataLoader(val_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)

test_dataset = Multidata(data_hsi, data_lidar, test_gt, windowSize)
test_loader = torch.utils.data.DataLoader(test_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)



KAPPA = []
OA = []
AA = []
ELEMENT_ACC = np.zeros((itm, num_classes))
training_time = []
testing_time = []


for itera in range(0, itm):
    
#     print("---------------------------------- Model Summary ---------------------------------------------")
    model = MFT(FM=FM, NC=hsi_band, NCLidar=lidar_band, Classes=num_classes, ntokens=ntokens, token_type='channel', 
                num_heads=num_heads, mlp_dim=mlp_dim, depth=depth).cuda()
    summary(model,[(hsi_band, windowSize,windowSize),(lidar_band, windowSize,windowSize)]) 

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4,weight_decay=5e-3)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.9)
    loss_func = nn.CrossEntropyLoss()
    
    model, train_time = train(model, loss_func, DEVICE, train_loader, optimizer, scheduler, EPOCH, val_loader, itera)
    test_acc_temp, test_loss_temp, y_pred, target, test_time = test(model, DEVICE, test_loader)
    oa,aa,kappa,each_acc, accuracy_matrix = reports(y_pred, target)
    
    input_1 = torch.randn(1, hsi_band, windowSize, windowSize)
    input_2 = torch.randn(1, lidar_band, windowSize, windowSize)
    input_1 = input_1.cuda()
    input_2 = input_2.cuda()
    macs, params = profile(model, inputs=(input_1,input_2, ))
    print("params, macs", params, macs)
    
    training_time.append(train_time)
    testing_time.append(test_time)
    KAPPA.append(kappa)
    OA.append(oa)
    AA.append(aa)
    ELEMENT_ACC[itera, :] = each_acc
    
    record.record_output(OA, AA, KAPPA, ELEMENT_ACC, training_time, testing_time, macs, params,train_sample, './' + FileName + '/' +
                         datasetName  + '_' + str(train_size) + '_' + str(params) + '_Report' +'.txt')

    print("final test results :", accuracy_matrix)

    del model,input_1,input_2


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv3d-1       [-1, 8, 136, 11, 11]             656
       BatchNorm3d-2       [-1, 8, 136, 11, 11]              16
              ReLU-3       [-1, 8, 136, 11, 11]               0
            Conv2d-4           [-1, 64, 11, 11]          39,232
            Conv2d-5           [-1, 64, 11, 11]          69,696
           HetConv-6           [-1, 64, 11, 11]               0
       BatchNorm2d-7           [-1, 64, 11, 11]             128
              ReLU-8           [-1, 64, 11, 11]               0
            Conv2d-9           [-1, 64, 11, 11]             640
      BatchNorm2d-10           [-1, 64, 11, 11]             128
             GELU-11           [-1, 64, 11, 11]               0
          Dropout-12                [-1, 5, 64]               0
        LayerNorm-13                [-1, 5, 64]             128
           Linear-14             [-1, 1

epoch 28, train loss 1.129898, train acc 0.607, valida  loss 1.013286, valida acc 0.671
epoch 29, train loss 1.168934, train acc 0.604, valida  loss 0.985779, valida acc 0.702
Best_Val_Value changed: from 0.702222 to 0.702222;	Best Classification Accuracy 0.702222， Best Classification loss 0.985779； Best Epoch： 29
epoch 30, train loss 1.059395, train acc 0.644, valida  loss 0.994771, valida acc 0.691
epoch 31, train loss 0.981620, train acc 0.651, valida  loss 0.882719, valida acc 0.718
Best_Val_Value changed: from 0.702222 to 0.717778;	Best Classification Accuracy 0.717778， Best Classification loss 0.882719； Best Epoch： 31
epoch 32, train loss 1.028441, train acc 0.669, valida  loss 0.884419, valida acc 0.758
Best_Val_Value changed: from 0.717778 to 0.757778;	Best Classification Accuracy 0.757778， Best Classification loss 0.884419； Best Epoch： 32
epoch 33, train loss 1.036860, train acc 0.676, valida  loss 0.843326, valida acc 0.687
epoch 34, train loss 0.931615, train acc 0.691, vali

epoch 99, train loss 0.576898, train acc 0.869, valida  loss 0.543086, valida acc 0.844
epoch 100, train loss 0.480772, train acc 0.851, valida  loss 0.482522, valida acc 0.860
epoch 101, train loss 0.479138, train acc 0.822, valida  loss 0.447293, valida acc 0.822
Counter 1 of 20
epoch 102, train loss 0.672889, train acc 0.869, valida  loss 0.458038, valida acc 0.838
Counter 2 of 20
epoch 103, train loss 0.429151, train acc 0.844, valida  loss 0.453430, valida acc 0.856
Counter 3 of 20
epoch 104, train loss 0.428100, train acc 0.867, valida  loss 0.439415, valida acc 0.856
Counter 4 of 20
epoch 105, train loss 0.606037, train acc 0.853, valida  loss 0.387330, valida acc 0.880
Best_Val_Value changed: from 0.880000 to 0.880000;	Best Classification Accuracy 0.880000， Best Classification loss 0.387330； Best Epoch： 105
epoch 106, train loss 0.570519, train acc 0.816, valida  loss 0.465724, valida acc 0.831
Counter 1 of 20
epoch 107, train loss 0.453137, train acc 0.844, valida  loss 0.5011

epoch 1, train loss 3.076088, train acc 0.076, valida  loss 2.717057, valida acc 0.098
Best_Val_Value changed: from 0.000000 to 0.097778;	Best Classification Accuracy 0.097778， Best Classification loss 2.717057； Best Epoch： 1
epoch 2, train loss 2.779967, train acc 0.107, valida  loss 2.601903, valida acc 0.062
epoch 3, train loss 2.602903, train acc 0.111, valida  loss 2.551385, valida acc 0.124
Best_Val_Value changed: from 0.097778 to 0.124444;	Best Classification Accuracy 0.124444， Best Classification loss 2.551385； Best Epoch： 3
epoch 4, train loss 2.581142, train acc 0.169, valida  loss 2.463289, valida acc 0.176
Best_Val_Value changed: from 0.124444 to 0.175556;	Best Classification Accuracy 0.175556， Best Classification loss 2.463289； Best Epoch： 4
epoch 5, train loss 2.472976, train acc 0.213, valida  loss 2.337763, valida acc 0.329
Best_Val_Value changed: from 0.175556 to 0.328889;	Best Classification Accuracy 0.328889， Best Classification loss 2.337763； Best Epoch： 5
epoch 6, 

epoch 57, train loss 0.774501, train acc 0.747, valida  loss 0.712165, valida acc 0.747
Best_Val_Value changed: from 0.746667 to 0.746667;	Best Classification Accuracy 0.746667， Best Classification loss 0.712165； Best Epoch： 57
epoch 58, train loss 0.772819, train acc 0.738, valida  loss 0.743824, valida acc 0.740
epoch 59, train loss 0.756134, train acc 0.751, valida  loss 0.751809, valida acc 0.753
Best_Val_Value changed: from 0.746667 to 0.753333;	Best Classification Accuracy 0.753333， Best Classification loss 0.751809； Best Epoch： 59
epoch 60, train loss 0.922629, train acc 0.727, valida  loss 0.777288, valida acc 0.727
epoch 61, train loss 0.731063, train acc 0.749, valida  loss 0.765451, valida acc 0.742
epoch 62, train loss 0.756701, train acc 0.760, valida  loss 0.737232, valida acc 0.751
epoch 63, train loss 0.693664, train acc 0.789, valida  loss 0.696333, valida acc 0.778
Best_Val_Value changed: from 0.753333 to 0.777778;	Best Classification Accuracy 0.777778， Best Classific

...

epoch 50, train loss 1.254057, train acc 0.716, valida  loss 0.687818, valida acc 0.811
epoch 51, train loss 0.747411, train acc 0.773, valida  loss 0.711246, valida acc 0.789
epoch 52, train loss 0.770602, train acc 0.802, valida  loss 0.635243, valida acc 0.802
epoch 53, train loss 0.715770, train acc 0.769, valida  loss 0.603232, valida acc 0.842
Best_Val_Value changed: from 0.840000 to 0.842222;	Best Classification Accuracy 0.842222， Best Classification loss 0.603232； Best Epoch： 53
epoch 54, train loss 0.737840, train acc 0.793, valida  loss 0.574569, valida acc 0.798
epoch 55, train loss 0.824708, train acc 0.787, valida  loss 0.583004, valida acc 0.836
epoch 56, train loss 0.763384, train acc 0.824, valida  loss 0.540391, valida acc 0.840
epoch 57, train loss 0.718686, train acc 0.773, valida  loss 0.610827, valida acc 0.820
epoch 58, train loss 0.791764, train acc 0.778, valida  loss 0.604958, valida acc 0.809
epoch 59, train loss 0.652158, train acc 0.798, valida  loss 0.55981


Test set: Average loss: 0.0000, Accuracy: 12566/14129 (88.9376%)
||======= Test Time for 0:00:17.030180 ======||
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv3d'>.
[INFO] Register count_bn() for <class 'torch.nn.modules.batchnorm.BatchNorm3d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[WARN] Cannot find rule for <class 'torch.nn.modules.container.Sequential'>. Treat it as zero Macs and zero Params.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[WARN] Cannot find rule for <class 'mft_model.HetConv'>. Treat it as zero Macs and zero Params.
[INFO] Register count_bn() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[WARN] Cannot find rule for <class 'torch.nn.modules.activation.GELU'>. Treat it as zero Macs and zero Params.
[WARN] Cannot find rule for <class 'torch.nn.modules.normalization.LayerNorm'>. Treat it as zero Macs and zero Params.
[INFO] Register count_linear() for <class 'torch.nn.modul

epoch 19, train loss 1.434780, train acc 0.460, valida  loss 1.420138, valida acc 0.516
Best_Val_Value changed: from 0.495556 to 0.515556;	Best Classification Accuracy 0.515556， Best Classification loss 1.420138； Best Epoch： 19
epoch 20, train loss 1.442487, train acc 0.489, valida  loss 1.312261, valida acc 0.527
Best_Val_Value changed: from 0.515556 to 0.526667;	Best Classification Accuracy 0.526667， Best Classification loss 1.312261； Best Epoch： 20
epoch 21, train loss 1.416709, train acc 0.511, valida  loss 1.357128, valida acc 0.509
epoch 22, train loss 1.494642, train acc 0.471, valida  loss 1.335872, valida acc 0.476
epoch 23, train loss 1.535982, train acc 0.502, valida  loss 1.230010, valida acc 0.567
Best_Val_Value changed: from 0.526667 to 0.566667;	Best Classification Accuracy 0.566667， Best Classification loss 1.230010； Best Epoch： 23
epoch 24, train loss 1.362557, train acc 0.516, valida  loss 1.272489, valida acc 0.553
epoch 25, train loss 1.319753, train acc 0.540, vali

epoch 81, train loss 0.862040, train acc 0.691, valida  loss 0.847690, valida acc 0.671
epoch 82, train loss 0.807611, train acc 0.707, valida  loss 0.817749, valida acc 0.680
epoch 83, train loss 0.775029, train acc 0.753, valida  loss 0.818765, valida acc 0.716
epoch 84, train loss 0.752726, train acc 0.720, valida  loss 0.746125, valida acc 0.767
Best_Val_Value changed: from 0.764444 to 0.766667;	Best Classification Accuracy 0.766667， Best Classification loss 0.746125； Best Epoch： 84
epoch 85, train loss 0.903490, train acc 0.731, valida  loss 0.746936, valida acc 0.727
epoch 86, train loss 0.809915, train acc 0.731, valida  loss 0.750201, valida acc 0.731
epoch 87, train loss 0.884932, train acc 0.704, valida  loss 0.725143, valida acc 0.713
epoch 88, train loss 0.723019, train acc 0.736, valida  loss 0.715489, valida acc 0.749
epoch 89, train loss 0.722084, train acc 0.762, valida  loss 0.722501, valida acc 0.747
epoch 90, train loss 0.708158, train acc 0.716, valida  loss 0.69815

epoch 148, train loss 0.518962, train acc 0.804, valida  loss 0.566090, valida acc 0.811
Counter 2 of 20
epoch 149, train loss 0.732272, train acc 0.833, valida  loss 0.476542, valida acc 0.842
Best_Val_Value changed: from 0.837778 to 0.842222;	Best Classification Accuracy 0.842222， Best Classification loss 0.476542； Best Epoch： 149
epoch 150, train loss 0.588753, train acc 0.836, valida  loss 0.497632, valida acc 0.827
Counter 1 of 20
epoch 151, train loss 0.641112, train acc 0.811, valida  loss 0.500320, valida acc 0.827
Counter 2 of 20
epoch 152, train loss 0.475819, train acc 0.836, valida  loss 0.587742, valida acc 0.789
Counter 3 of 20
epoch 153, train loss 0.502215, train acc 0.844, valida  loss 0.593386, valida acc 0.807
Counter 4 of 20
epoch 154, train loss 0.512764, train acc 0.822, valida  loss 0.558322, valida acc 0.844
Best_Val_Value changed: from 0.842222 to 0.844444;	Best Classification Accuracy 0.844444， Best Classification loss 0.558322； Best Epoch： 154
epoch 155, trai

In [7]:
sys.path.append("./exvit/")
from MViT_pytorch_upload import *

windowSize = 13
BATCH_SIZE = 64
EPOCH = 500
FileName = 'exvit'
norm_type = 'bandwise'
band_MultiModal = [hsi_band, lidar_band]


data_hsi = mynorm(data_hsi_o, norm_type)
data_lidar = data_lidar_o


train_dataset = Multidata(data_hsi, data_lidar, train_gt, windowSize)
train_loader = torch.utils.data.DataLoader(train_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)

val_dataset = Multidata(data_hsi, data_lidar, val_gt, windowSize)
val_loader = torch.utils.data.DataLoader(val_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)

test_dataset = Multidata(data_hsi, data_lidar, test_gt, windowSize)
test_loader = torch.utils.data.DataLoader(test_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)


KAPPA = []
OA = []
AA = []
ELEMENT_ACC = np.zeros((itm, num_classes))
training_time = []
testing_time = []


for itera in range(0, itm):
    
#     print("---------------------------------- Model Summary ---------------------------------------------")
    model = MViT(
        patch_size = windowSize,
        num_patches = band_MultiModal,
        num_classes = num_classes,
        dim = 64,
        depth = 6,
        heads = 4,
        mlp_dim = 32,
        dropout = 0.1,
        emb_dropout = 0.1,
        mode = 'MViT'
    )
    model = model.cuda()
    summary(model,[(hsi_band, windowSize,windowSize),(lidar_band, windowSize,windowSize)]) 

    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4,weight_decay=0.)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.9)
    loss_func = nn.CrossEntropyLoss()
    
    model, train_time = train(model, loss_func, DEVICE, train_loader, optimizer, scheduler, EPOCH, val_loader, itera)
    test_acc_temp, test_loss_temp, y_pred, target, test_time = test(model, DEVICE, test_loader)
    oa,aa,kappa,each_acc, accuracy_matrix = reports(y_pred, target)
    
    input_1 = torch.randn(1, hsi_band, windowSize, windowSize)
    input_2 = torch.randn(1, lidar_band, windowSize, windowSize)
    input_1 = input_1.cuda()
    input_2 = input_2.cuda()
    macs, params = profile(model, inputs=(input_1,input_2, ))
    print("params, macs", params, macs)
    
    training_time.append(train_time)
    testing_time.append(test_time)
    KAPPA.append(kappa)
    OA.append(oa)
    AA.append(aa)
    ELEMENT_ACC[itera, :] = each_acc
    
    record.record_output(OA, AA, KAPPA, ELEMENT_ACC, training_time, testing_time, macs, params,train_sample, './' + FileName + '/' +
                         datasetName  + '_' + str(train_size) + '_' + str(params) + '_Report' +'.txt')

    print("final test results :", accuracy_matrix)

    del model,input_1,input_2

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1          [-1, 144, 13, 13]           1,440
            Conv2d-2           [-1, 16, 13, 13]           2,320
       BatchNorm2d-3           [-1, 16, 13, 13]              32
              GELU-4           [-1, 16, 13, 13]               0
            Conv2d-5           [-1, 16, 13, 13]             160
            Conv2d-6           [-1, 32, 13, 13]             544
       BatchNorm2d-7           [-1, 32, 13, 13]              64
              GELU-8           [-1, 32, 13, 13]               0
            Conv2d-9           [-1, 32, 13, 13]             320
           Conv2d-10           [-1, 64, 13, 13]           2,112
      BatchNorm2d-11           [-1, 64, 13, 13]             128
             GELU-12           [-1, 64, 13, 13]               0
           Linear-13              [-1, 169, 64]           4,160
          Dropout-14              [-1, 

epoch 1, train loss 2.650818, train acc 0.149, valida  loss 2.698892, valida acc 0.098
Best_Val_Value changed: from 0.000000 to 0.097778;	Best Classification Accuracy 0.097778， Best Classification loss 2.698892； Best Epoch： 1
epoch 2, train loss 2.236673, train acc 0.320, valida  loss 2.711776, valida acc 0.104
Best_Val_Value changed: from 0.097778 to 0.104444;	Best Classification Accuracy 0.104444， Best Classification loss 2.711776； Best Epoch： 2
epoch 3, train loss 1.902307, train acc 0.420, valida  loss 2.764853, valida acc 0.124
Best_Val_Value changed: from 0.104444 to 0.124444;	Best Classification Accuracy 0.124444， Best Classification loss 2.764853； Best Epoch： 3
epoch 4, train loss 1.671783, train acc 0.473, valida  loss 2.651831, valida acc 0.149
Best_Val_Value changed: from 0.124444 to 0.148889;	Best Classification Accuracy 0.148889， Best Classification loss 2.651831； Best Epoch： 4
epoch 5, train loss 1.671028, train acc 0.513, valida  loss 2.406755, valida acc 0.204
Best_Val_

epoch 66, train loss 0.407099, train acc 0.913, valida  loss 0.677625, valida acc 0.791
epoch 67, train loss 0.314855, train acc 0.947, valida  loss 1.038814, valida acc 0.711
epoch 68, train loss 0.852629, train acc 0.924, valida  loss 0.509507, valida acc 0.811
epoch 69, train loss 0.357941, train acc 0.936, valida  loss 0.419456, valida acc 0.882
epoch 70, train loss 0.431184, train acc 0.924, valida  loss 1.091630, valida acc 0.713
epoch 71, train loss 0.711728, train acc 0.911, valida  loss 1.016234, valida acc 0.700
epoch 72, train loss 0.679322, train acc 0.869, valida  loss 1.268651, valida acc 0.658
epoch 73, train loss 0.614367, train acc 0.860, valida  loss 0.599496, valida acc 0.813
epoch 74, train loss 0.607581, train acc 0.887, valida  loss 0.581888, valida acc 0.824
epoch 75, train loss 0.360162, train acc 0.909, valida  loss 0.745502, valida acc 0.740
epoch 76, train loss 0.613531, train acc 0.900, valida  loss 0.766177, valida acc 0.813
epoch 77, train loss 0.295425, t

epoch 1, train loss 2.666323, train acc 0.189, valida  loss 2.725549, valida acc 0.084
Best_Val_Value changed: from 0.000000 to 0.084444;	Best Classification Accuracy 0.084444， Best Classification loss 2.725549； Best Epoch： 1
epoch 2, train loss 2.226011, train acc 0.344, valida  loss 2.904430, valida acc 0.038
epoch 3, train loss 2.064017, train acc 0.433, valida  loss 2.816785, valida acc 0.082
epoch 4, train loss 1.718546, train acc 0.533, valida  loss 2.771487, valida acc 0.049
epoch 5, train loss 1.508133, train acc 0.578, valida  loss 2.316983, valida acc 0.162
Best_Val_Value changed: from 0.084444 to 0.162222;	Best Classification Accuracy 0.162222， Best Classification loss 2.316983； Best Epoch： 5
epoch 6, train loss 1.415817, train acc 0.547, valida  loss 2.152893, valida acc 0.291
Best_Val_Value changed: from 0.162222 to 0.291111;	Best Classification Accuracy 0.291111， Best Classification loss 2.152893； Best Epoch： 6
epoch 7, train loss 1.380721, train acc 0.598, valida  loss 2

epoch 74, train loss 0.469129, train acc 0.889, valida  loss 1.206703, valida acc 0.649
epoch 75, train loss 0.661797, train acc 0.871, valida  loss 0.765274, valida acc 0.780
epoch 76, train loss 0.579835, train acc 0.871, valida  loss 0.696259, valida acc 0.811
epoch 77, train loss 0.425716, train acc 0.896, valida  loss 0.725266, valida acc 0.773
epoch 78, train loss 0.440489, train acc 0.873, valida  loss 0.580417, valida acc 0.811
epoch 79, train loss 0.669773, train acc 0.898, valida  loss 0.609713, valida acc 0.813
epoch 80, train loss 0.418369, train acc 0.856, valida  loss 1.319036, valida acc 0.573
epoch 81, train loss 0.710494, train acc 0.856, valida  loss 0.714441, valida acc 0.780
epoch 82, train loss 0.579093, train acc 0.878, valida  loss 0.474869, valida acc 0.860
Best_Val_Value changed: from 0.857778 to 0.860000;	Best Classification Accuracy 0.860000， Best Classification loss 0.474869； Best Epoch： 82
epoch 83, train loss 0.772587, train acc 0.887, valida  loss 0.76997

...

epoch 1, train loss 2.505684, train acc 0.182, valida  loss 2.645230, valida acc 0.084
Best_Val_Value changed: from 0.000000 to 0.084444;	Best Classification Accuracy 0.084444， Best Classification loss 2.645230； Best Epoch： 1
epoch 2, train loss 2.199260, train acc 0.213, valida  loss 2.681228, valida acc 0.082
epoch 3, train loss 1.905945, train acc 0.369, valida  loss 2.840506, valida acc 0.082
epoch 4, train loss 1.827464, train acc 0.496, valida  loss 2.507501, valida acc 0.153
Best_Val_Value changed: from 0.084444 to 0.153333;	Best Classification Accuracy 0.153333， Best Classification loss 2.507501； Best Epoch： 4
epoch 5, train loss 1.781890, train acc 0.507, valida  loss 2.450760, valida acc 0.144
epoch 6, train loss 1.543785, train acc 0.553, valida  loss 2.264484, valida acc 0.244
Best_Val_Value changed: from 0.153333 to 0.244444;	Best Classification Accuracy 0.244444， Best Classification loss 2.264484； Best Epoch： 6
epoch 7, train loss 1.571801, train acc 0.587, valida  loss 2

epoch 72, train loss 0.694076, train acc 0.856, valida  loss 0.557379, valida acc 0.829
epoch 73, train loss 0.428415, train acc 0.907, valida  loss 0.621386, valida acc 0.762
epoch 74, train loss 0.477805, train acc 0.891, valida  loss 0.707587, valida acc 0.789
epoch 75, train loss 0.469737, train acc 0.896, valida  loss 1.334093, valida acc 0.720
epoch 76, train loss 0.422675, train acc 0.909, valida  loss 0.532034, valida acc 0.816
epoch 77, train loss 0.425902, train acc 0.898, valida  loss 0.589786, valida acc 0.800
epoch 78, train loss 0.454216, train acc 0.902, valida  loss 0.586401, valida acc 0.787
epoch 79, train loss 0.486751, train acc 0.902, valida  loss 0.748860, valida acc 0.793
epoch 80, train loss 0.414639, train acc 0.893, valida  loss 0.582895, valida acc 0.831
epoch 81, train loss 0.818639, train acc 0.907, valida  loss 0.394651, valida acc 0.867
Best_Val_Value changed: from 0.837778 to 0.866667;	Best Classification Accuracy 0.866667， Best Classification loss 0.394

epoch 144, train loss 0.500784, train acc 0.918, valida  loss 0.354824, valida acc 0.862
Counter 5 of 20
epoch 145, train loss 0.200216, train acc 0.949, valida  loss 0.431343, valida acc 0.902
Counter 6 of 20
epoch 146, train loss 0.219212, train acc 0.953, valida  loss 0.340530, valida acc 0.900
Counter 7 of 20
epoch 147, train loss 0.396345, train acc 0.951, valida  loss 0.414638, valida acc 0.900
Counter 8 of 20
epoch 148, train loss 0.240565, train acc 0.933, valida  loss 0.443867, valida acc 0.856
Counter 9 of 20
epoch 149, train loss 0.575548, train acc 0.938, valida  loss 0.712539, valida acc 0.831
Counter 10 of 20
epoch 150, train loss 0.297015, train acc 0.953, valida  loss 0.421377, valida acc 0.876
Counter 11 of 20
epoch 151, train loss 0.305179, train acc 0.938, valida  loss 0.349750, valida acc 0.882
Counter 12 of 20
epoch 152, train loss 0.216775, train acc 0.938, valida  loss 0.384450, valida acc 0.847
Counter 13 of 20
epoch 153, train loss 0.384039, train acc 0.922, va

epoch 1, train loss 2.757218, train acc 0.102, valida  loss 2.610507, valida acc 0.102
Best_Val_Value changed: from 0.000000 to 0.102222;	Best Classification Accuracy 0.102222， Best Classification loss 2.610507； Best Epoch： 1
epoch 2, train loss 2.388977, train acc 0.298, valida  loss 2.638447, valida acc 0.142
Best_Val_Value changed: from 0.102222 to 0.142222;	Best Classification Accuracy 0.142222， Best Classification loss 2.638447； Best Epoch： 2
epoch 3, train loss 2.114567, train acc 0.358, valida  loss 2.535968, valida acc 0.131
epoch 4, train loss 1.717568, train acc 0.491, valida  loss 2.535996, valida acc 0.147
Best_Val_Value changed: from 0.142222 to 0.146667;	Best Classification Accuracy 0.146667， Best Classification loss 2.535996； Best Epoch： 4
epoch 5, train loss 1.666679, train acc 0.533, valida  loss 2.387445, valida acc 0.169
Best_Val_Value changed: from 0.146667 to 0.168889;	Best Classification Accuracy 0.168889， Best Classification loss 2.387445； Best Epoch： 5
epoch 6, 

epoch 71, train loss 0.555183, train acc 0.891, valida  loss 1.784833, valida acc 0.487
epoch 72, train loss 0.599055, train acc 0.911, valida  loss 0.723905, valida acc 0.780
epoch 73, train loss 0.601443, train acc 0.918, valida  loss 0.684992, valida acc 0.807
epoch 74, train loss 0.439029, train acc 0.909, valida  loss 0.498047, valida acc 0.864
Best_Val_Value changed: from 0.846667 to 0.864444;	Best Classification Accuracy 0.864444， Best Classification loss 0.498047； Best Epoch： 74
epoch 75, train loss 0.523397, train acc 0.902, valida  loss 0.653910, valida acc 0.793
epoch 76, train loss 0.686381, train acc 0.902, valida  loss 0.511829, valida acc 0.822
epoch 77, train loss 0.468785, train acc 0.893, valida  loss 0.477804, valida acc 0.822
epoch 78, train loss 0.354664, train acc 0.898, valida  loss 0.886107, valida acc 0.704
epoch 79, train loss 0.459882, train acc 0.918, valida  loss 0.488138, valida acc 0.847
epoch 80, train loss 0.410680, train acc 0.933, valida  loss 0.67547

In [11]:
sys.path.append("./amsse/")
from model_amsse import *
from tool_amsse_n import *
import involution

# dataset = ['Trento', 'MUUFL', 'Houston']
# batch_sizes = [16, 128, 32]
# windowSize = [17, 13, 11]

windowSize = 11
BATCH_SIZE = 32
EPOCH = 200
FileName = 'amsse'

HSI_data, _ = applyPCA_here(data_hsi_o, numComponents=15)

if datasetName == 'muufl':
    data_lidar, _ = applyPCA_here(data_lidar_o, numComponents=2)
    data_lidar = normalization_here(data_lidar, type=1)
else:
    data_lidar, _ = applyPCA_here(data_lidar_o, numComponents=1)
    data_lidar = normalization_here(data_lidar, type=1)

data_hsi = normalization_here(HSI_data, type=1)


train_dataset = Multidata(data_hsi, data_lidar, train_gt, windowSize)
train_loader = torch.utils.data.DataLoader(train_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)

val_dataset = Multidata(data_hsi, data_lidar, val_gt, windowSize)
val_loader = torch.utils.data.DataLoader(val_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)

test_dataset = Multidata(data_hsi, data_lidar, test_gt, windowSize)
test_loader = torch.utils.data.DataLoader(test_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)


KAPPA = []
OA = []
AA = []
ELEMENT_ACC = np.zeros((itm, num_classes))
training_time = []
testing_time = []


for itera in range(0, itm):
    
#     print("---------------------------------- Model Summary ---------------------------------------------")
    model = fusion_main(15, lidar_band, num_classes, windowSize).cuda()
    summary(model,[(15, windowSize,windowSize),(lidar_band, windowSize,windowSize)]) 

    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=80, gamma=0.5)
    loss_func = nn.CrossEntropyLoss()
    
    model, train_time = trainamsse(model, loss_func, DEVICE, train_loader, optimizer, scheduler, EPOCH, val_loader, itera)
    test_acc_temp, test_loss_temp, y_pred, target, test_time = testamsse(model, DEVICE, test_loader)
    oa,aa,kappa,each_acc, accuracy_matrix = reports(y_pred, target)
    
    input_1 = torch.randn(1, 15, windowSize, windowSize)
    input_2 = torch.randn(1, lidar_band, windowSize, windowSize)
    input_1 = input_1.cuda()
    input_2 = input_2.cuda()
    macs, params = profile(model, inputs=(input_1,input_2, ))
    print("params, macs", params, macs)
    
    training_time.append(train_time)
    testing_time.append(test_time)
    KAPPA.append(kappa)
    OA.append(oa)
    AA.append(aa)
    ELEMENT_ACC[itera, :] = each_acc
    
    record.record_output(OA, AA, KAPPA, ELEMENT_ACC, training_time, testing_time, macs, params,train_sample, './' + FileName + '/' +
                         datasetName  + '_' + str(train_size) + '_' + str(params) + '_Report' +'.txt')

    print("final test results :", accuracy_matrix)

    del model,input_1,input_2


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 16, 11, 11]             240
            Conv2d-2           [-1, 32, 11, 11]             512
            Conv2d-3           [-1, 64, 11, 11]           2,048
       Feature_HSI-4           [-1, 64, 11, 11]               0
            Conv2d-5            [-1, 3, 11, 11]              45
       BatchNorm2d-6            [-1, 3, 11, 11]               6
              ReLU-7            [-1, 3, 11, 11]               0
        ConvModule-8            [-1, 3, 11, 11]               0
            Conv2d-9           [-1, 45, 11, 11]             180
             ReLU-10           [-1, 45, 11, 11]               0
       ConvModule-11           [-1, 45, 11, 11]               0
       involution-12           [-1, 15, 11, 11]               0
           Conv2d-13           [-1, 64, 11, 11]             960
      BatchNorm2d-14           [-1, 64,

epoch 17, train loss 1.416648, train acc 0.589, valida  loss 1.638059, valida acc 0.567
epoch 18, train loss 1.167209, train acc 0.660, valida  loss 1.409685, valida acc 0.538
epoch 19, train loss 1.036720, train acc 0.709, valida  loss 0.917545, valida acc 0.711
epoch 20, train loss 0.994918, train acc 0.709, valida  loss 1.271118, valida acc 0.667
epoch 21, train loss 0.922170, train acc 0.736, valida  loss 0.824239, valida acc 0.731
epoch 22, train loss 0.927299, train acc 0.709, valida  loss 2.122470, valida acc 0.576
epoch 23, train loss 1.037048, train acc 0.720, valida  loss 1.146371, valida acc 0.678
epoch 24, train loss 0.876818, train acc 0.762, valida  loss 0.717238, valida acc 0.802
Best_Val_Value changed: from 0.773333 to 0.802222;	Best Classification Accuracy 0.802222， Best Classification loss 0.717238； Best Epoch： 24
epoch 25, train loss 0.827318, train acc 0.773, valida  loss 0.699618, valida acc 0.787
epoch 26, train loss 0.845363, train acc 0.758, valida  loss 0.76777

epoch 88, train loss 0.285733, train acc 0.960, valida  loss 0.382860, valida acc 0.900
epoch 89, train loss 0.251015, train acc 0.960, valida  loss 0.418174, valida acc 0.876
epoch 90, train loss 0.283958, train acc 0.956, valida  loss 0.394958, valida acc 0.896
epoch 91, train loss 0.313052, train acc 0.940, valida  loss 0.362856, valida acc 0.904
epoch 92, train loss 0.441115, train acc 0.956, valida  loss 0.399822, valida acc 0.891
epoch 93, train loss 0.403751, train acc 0.969, valida  loss 0.488970, valida acc 0.896
epoch 94, train loss 0.270826, train acc 0.973, valida  loss 0.356314, valida acc 0.893
epoch 95, train loss 0.426370, train acc 0.956, valida  loss 0.391044, valida acc 0.884
epoch 96, train loss 0.268632, train acc 0.960, valida  loss 0.378416, valida acc 0.887
epoch 97, train loss 0.304509, train acc 0.949, valida  loss 0.515072, valida acc 0.876
epoch 98, train loss 0.262134, train acc 0.960, valida  loss 0.386669, valida acc 0.909
epoch 99, train loss 0.255675, t

epoch 1, train loss 2.430025, train acc 0.264, valida  loss 2.588747, valida acc 0.336
Best_Val_Value changed: from 0.000000 to 0.335556;	Best Classification Accuracy 0.335556， Best Classification loss 2.588747； Best Epoch： 1
epoch 2, train loss 2.097048, train acc 0.380, valida  loss 1.940429, valida acc 0.338
Best_Val_Value changed: from 0.335556 to 0.337778;	Best Classification Accuracy 0.337778， Best Classification loss 1.940429； Best Epoch： 2
epoch 3, train loss 1.869185, train acc 0.476, valida  loss 1.615044, valida acc 0.533
Best_Val_Value changed: from 0.337778 to 0.533333;	Best Classification Accuracy 0.533333， Best Classification loss 1.615044； Best Epoch： 3
epoch 4, train loss 1.680108, train acc 0.536, valida  loss 1.387506, valida acc 0.582
Best_Val_Value changed: from 0.533333 to 0.582222;	Best Classification Accuracy 0.582222， Best Classification loss 1.387506； Best Epoch： 4
epoch 5, train loss 1.567278, train acc 0.587, valida  loss 1.247783, valida acc 0.611
Best_Val_

epoch 66, train loss 0.233606, train acc 0.967, valida  loss 0.307905, valida acc 0.940
epoch 67, train loss 0.255222, train acc 0.953, valida  loss 0.266760, valida acc 0.913
epoch 68, train loss 0.225317, train acc 0.976, valida  loss 0.249225, valida acc 0.922
epoch 69, train loss 0.354012, train acc 0.960, valida  loss 0.324601, valida acc 0.918
epoch 70, train loss 0.236823, train acc 0.958, valida  loss 0.222905, valida acc 0.936
epoch 71, train loss 0.266879, train acc 0.969, valida  loss 0.280751, valida acc 0.916
epoch 72, train loss 0.244654, train acc 0.964, valida  loss 0.277364, valida acc 0.909
epoch 73, train loss 0.281598, train acc 0.960, valida  loss 0.266716, valida acc 0.931
epoch 74, train loss 0.291201, train acc 0.944, valida  loss 0.480943, valida acc 0.924
epoch 75, train loss 0.478743, train acc 0.911, valida  loss 0.276986, valida acc 0.922
epoch 76, train loss 0.290124, train acc 0.960, valida  loss 0.242628, valida acc 0.922
epoch 77, train loss 0.269056, t

...

epoch 66, train loss 0.345992, train acc 0.936, valida  loss 0.467635, valida acc 0.851
epoch 67, train loss 0.310834, train acc 0.960, valida  loss 0.452291, valida acc 0.878
epoch 68, train loss 0.444375, train acc 0.933, valida  loss 0.374974, valida acc 0.878
epoch 69, train loss 0.402310, train acc 0.940, valida  loss 0.408714, valida acc 0.896
Best_Val_Value changed: from 0.893333 to 0.895556;	Best Classification Accuracy 0.895556， Best Classification loss 0.408714； Best Epoch： 69
epoch 70, train loss 0.322412, train acc 0.956, valida  loss 0.367494, valida acc 0.896
Best_Val_Value changed: from 0.895556 to 0.895556;	Best Classification Accuracy 0.895556， Best Classification loss 0.367494； Best Epoch： 70
epoch 71, train loss 0.302492, train acc 0.947, valida  loss 0.501217, valida acc 0.907
Best_Val_Value changed: from 0.895556 to 0.906667;	Best Classification Accuracy 0.906667， Best Classification loss 0.501217； Best Epoch： 71
epoch 72, train loss 0.343147, train acc 0.924, vali

epoch 1, train loss 2.558306, train acc 0.224, valida  loss 2.469286, valida acc 0.362
Best_Val_Value changed: from 0.000000 to 0.362222;	Best Classification Accuracy 0.362222， Best Classification loss 2.469286； Best Epoch： 1
epoch 2, train loss 2.093453, train acc 0.471, valida  loss 1.765602, valida acc 0.538
Best_Val_Value changed: from 0.362222 to 0.537778;	Best Classification Accuracy 0.537778， Best Classification loss 1.765602； Best Epoch： 2
epoch 3, train loss 1.692281, train acc 0.596, valida  loss 1.412762, valida acc 0.662
Best_Val_Value changed: from 0.537778 to 0.662222;	Best Classification Accuracy 0.662222， Best Classification loss 1.412762； Best Epoch： 3
epoch 4, train loss 1.412012, train acc 0.671, valida  loss 1.172786, valida acc 0.651
epoch 5, train loss 1.207564, train acc 0.736, valida  loss 1.073365, valida acc 0.711
Best_Val_Value changed: from 0.662222 to 0.711111;	Best Classification Accuracy 0.711111， Best Classification loss 1.073365； Best Epoch： 5
epoch 6, 

epoch 68, train loss 0.670219, train acc 0.847, valida  loss 0.450096, valida acc 0.893
epoch 69, train loss 0.495694, train acc 0.887, valida  loss 0.362173, valida acc 0.916
epoch 70, train loss 0.427908, train acc 0.911, valida  loss 0.367453, valida acc 0.902
epoch 71, train loss 0.460783, train acc 0.902, valida  loss 0.321338, valida acc 0.909
epoch 72, train loss 0.415532, train acc 0.940, valida  loss 0.281643, valida acc 0.922
epoch 73, train loss 0.451351, train acc 0.944, valida  loss 0.231570, valida acc 0.938
epoch 74, train loss 0.344626, train acc 0.962, valida  loss 0.373689, valida acc 0.929
epoch 75, train loss 0.337776, train acc 0.949, valida  loss 0.245392, valida acc 0.936
epoch 76, train loss 0.331329, train acc 0.967, valida  loss 0.232758, valida acc 0.933
epoch 77, train loss 0.287524, train acc 0.967, valida  loss 0.190352, valida acc 0.947
epoch 78, train loss 0.280636, train acc 0.960, valida  loss 0.316199, valida acc 0.940
epoch 79, train loss 0.336662, t

epoch 1, train loss 2.496720, train acc 0.207, valida  loss 2.714607, valida acc 0.242
Best_Val_Value changed: from 0.000000 to 0.242222;	Best Classification Accuracy 0.242222， Best Classification loss 2.714607； Best Epoch： 1
epoch 2, train loss 2.214636, train acc 0.351, valida  loss 1.920422, valida acc 0.416
Best_Val_Value changed: from 0.242222 to 0.415556;	Best Classification Accuracy 0.415556， Best Classification loss 1.920422； Best Epoch： 2
epoch 3, train loss 1.958962, train acc 0.451, valida  loss 1.638241, valida acc 0.518
Best_Val_Value changed: from 0.415556 to 0.517778;	Best Classification Accuracy 0.517778， Best Classification loss 1.638241； Best Epoch： 3
epoch 4, train loss 1.606269, train acc 0.600, valida  loss 1.571731, valida acc 0.598
Best_Val_Value changed: from 0.517778 to 0.597778;	Best Classification Accuracy 0.597778， Best Classification loss 1.571731； Best Epoch： 4
epoch 5, train loss 1.414669, train acc 0.629, valida  loss 1.378118, valida acc 0.593
epoch 6, 

epoch 61, train loss 0.355925, train acc 0.967, valida  loss 0.215304, valida acc 0.936
epoch 62, train loss 0.214773, train acc 0.969, valida  loss 0.247152, valida acc 0.920
epoch 63, train loss 0.327786, train acc 0.971, valida  loss 0.228366, valida acc 0.936
epoch 64, train loss 0.339456, train acc 0.960, valida  loss 0.203974, valida acc 0.938
epoch 65, train loss 0.245343, train acc 0.962, valida  loss 0.266635, valida acc 0.911
epoch 66, train loss 0.295806, train acc 0.958, valida  loss 0.222643, valida acc 0.940
epoch 67, train loss 0.249549, train acc 0.971, valida  loss 0.219511, valida acc 0.942
epoch 68, train loss 0.225782, train acc 0.967, valida  loss 0.194405, valida acc 0.938
epoch 69, train loss 0.185439, train acc 0.971, valida  loss 0.237512, valida acc 0.924
epoch 70, train loss 0.340647, train acc 0.964, valida  loss 0.310343, valida acc 0.900
epoch 71, train loss 0.244594, train acc 0.967, valida  loss 0.257355, valida acc 0.922
epoch 72, train loss 0.238437, t

In [9]:
sys.path.append("./macn/")
from network import *
from tool_macn import *


windowSize = 11
BATCH_SIZE = 64
EPOCH = 100
FileName = 'macn'
numComponents = 30

data_hsi, _ = applyPCA(data_hsi_o, numComponents=numComponents)
data = data_hsi.reshape(np.prod(data_hsi.shape[:2]), np.prod(data_hsi.shape[2:]))
data = preprocessing.scale(data)
data_hsi = data.reshape(data_hsi.shape[0], data_hsi.shape[1], data_hsi.shape[2])

data_2 = data_lidar_o.reshape(np.prod(data_lidar_o.shape[:2]), np.prod(data_lidar_o.shape[2:]))
data_2 = preprocessing.scale(data_2)
data_lidar = data_2.reshape(data_lidar_o.shape[0], data_lidar_o.shape[1],data_lidar_o.shape[2])



train_dataset = Multidata_macn(data_hsi, data_lidar, train_gt, windowSize)
train_loader = torch.utils.data.DataLoader(train_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)

val_dataset = Multidata_macn(data_hsi, data_lidar, val_gt, windowSize)
val_loader = torch.utils.data.DataLoader(val_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)

test_dataset = Multidata_macn(data_hsi, data_lidar, test_gt, windowSize)
test_loader = torch.utils.data.DataLoader(test_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)



KAPPA = []
OA = []
AA = []
ELEMENT_ACC = np.zeros((itm, num_classes))
training_time = []
testing_time = []


for itera in range(0, itm):
    
#     print("---------------------------------- Model Summary ---------------------------------------------")
    model = MixConvNet(num_classes=num_classes, in_channels=lidar_band).to(DEVICE)
#     summary(model,[(1, numComponents, windowSize,windowSize),(lidar_band, windowSize,windowSize)]) 

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=1)
    loss_func = FocalLoss(class_num=num_classes)
    
    model, train_time = train(model, loss_func, DEVICE, train_loader, optimizer, scheduler, EPOCH, val_loader, itera)
    test_acc_temp, test_loss_temp, y_pred, target, test_time = test(model, DEVICE, test_loader)
    oa,aa,kappa,each_acc, accuracy_matrix = reports(y_pred, target)
    
    input_1 = torch.randn(1, 1, numComponents, windowSize, windowSize)
    input_2 = torch.randn(1, lidar_band, windowSize, windowSize)
    input_1 = input_1.cuda()
    input_2 = input_2.cuda()
    macs, params = profile(model, inputs=(input_1,input_2, ))
    print("params, macs", params, macs)
    
    training_time.append(train_time)
    testing_time.append(test_time)
    KAPPA.append(kappa)
    OA.append(oa)
    AA.append(aa)
    ELEMENT_ACC[itera, :] = each_acc
    
    record.record_output(OA, AA, KAPPA, ELEMENT_ACC, training_time, testing_time, macs, params,train_sample, './' + FileName + '/' +
                         datasetName  + '_' + str(train_size) + '_' + str(params) + '_Report' +'.txt')

    print("final test results :", accuracy_matrix)

    del model,input_1,input_2


D:\anaconda3\envs\pytorch_envs\lib\site-packages\sklearn\preprocessing\_data.py:236: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  "Numerical issues were encountered "


epoch 1, train loss 0.568871, train acc 0.193, valida  loss 0.427720, valida acc 0.496
Best_Val_Value changed: from 0.000000 to 0.495556;	Best Classification Accuracy 0.495556， Best Classification loss 0.427720； Best Epoch： 1
epoch 2, train loss 0.348623, train acc 0.400, valida  loss 0.262549, valida acc 0.587
Best_Val_Value changed: from 0.495556 to 0.586667;	Best Classification Accuracy 0.586667， Best Classification loss 0.262549； Best Epoch： 2
epoch 3, train loss 0.238975, train acc 0.631, valida  loss 0.195300, valida acc 0.711
Best_Val_Value changed: from 0.586667 to 0.711111;	Best Classification Accuracy 0.711111， Best Classification loss 0.195300； Best Epoch： 3
epoch 4, train loss 0.187388, train acc 0.704, valida  loss 0.123587, valida acc 0.744
Best_Val_Value changed: from 0.711111 to 0.744444;	Best Classification Accuracy 0.744444， Best Classification loss 0.123587； Best Epoch： 4
epoch 5, train loss 0.182494, train acc 0.660, valida  loss 0.136997, valida acc 0.709
epoch 6, 

epoch 68, train loss 0.065101, train acc 0.944, valida  loss 0.027093, valida acc 0.927
Counter 13 of 20
epoch 69, train loss 0.054801, train acc 0.933, valida  loss 0.056008, valida acc 0.900
Counter 14 of 20
epoch 70, train loss 0.031677, train acc 0.958, valida  loss 0.035171, valida acc 0.909
Counter 15 of 20
epoch 71, train loss 0.059147, train acc 0.938, valida  loss 0.041887, valida acc 0.896
Counter 16 of 20
epoch 72, train loss 0.024218, train acc 0.938, valida  loss 0.029618, valida acc 0.931
Best_Val_Value changed: from 0.931111 to 0.931111;	Best Classification Accuracy 0.931111， Best Classification loss 0.029618； Best Epoch： 72
epoch 73, train loss 0.043945, train acc 0.938, valida  loss 0.072270, valida acc 0.911
Counter 1 of 20
epoch 74, train loss 0.052259, train acc 0.929, valida  loss 0.044383, valida acc 0.902
Counter 2 of 20
epoch 75, train loss 0.034724, train acc 0.896, valida  loss 0.052224, valida acc 0.813
Counter 3 of 20
epoch 76, train loss 0.094149, train acc

epoch 14, train loss 0.140973, train acc 0.860, valida  loss 0.096107, valida acc 0.862
Best_Val_Value changed: from 0.851111 to 0.862222;	Best Classification Accuracy 0.862222， Best Classification loss 0.096107； Best Epoch： 14
epoch 15, train loss 0.150553, train acc 0.780, valida  loss 0.105019, valida acc 0.767
epoch 16, train loss 0.187616, train acc 0.856, valida  loss 0.104443, valida acc 0.822
epoch 17, train loss 0.092034, train acc 0.856, valida  loss 0.080003, valida acc 0.827
epoch 18, train loss 0.138166, train acc 0.842, valida  loss 0.123143, valida acc 0.831
epoch 19, train loss 0.130209, train acc 0.844, valida  loss 0.071515, valida acc 0.856
epoch 20, train loss 0.113387, train acc 0.884, valida  loss 0.124166, valida acc 0.836
epoch 21, train loss 0.111599, train acc 0.864, valida  loss 0.067199, valida acc 0.869
Best_Val_Value changed: from 0.862222 to 0.868889;	Best Classification Accuracy 0.868889， Best Classification loss 0.067199； Best Epoch： 21
epoch 22, train 

epoch 1, train loss 0.551096, train acc 0.202, valida  loss 0.434080, valida acc 0.387
Best_Val_Value changed: from 0.000000 to 0.386667;	Best Classification Accuracy 0.386667， Best Classification loss 0.434080； Best Epoch： 1
epoch 2, train loss 0.341837, train acc 0.442, valida  loss 0.293105, valida acc 0.527
Best_Val_Value changed: from 0.386667 to 0.526667;	Best Classification Accuracy 0.526667， Best Classification loss 0.293105； Best Epoch： 2
epoch 3, train loss 0.283087, train acc 0.573, valida  loss 0.185330, valida acc 0.644
Best_Val_Value changed: from 0.526667 to 0.644444;	Best Classification Accuracy 0.644444， Best Classification loss 0.185330； Best Epoch： 3
epoch 4, train loss 0.198547, train acc 0.700, valida  loss 0.141309, valida acc 0.751
Best_Val_Value changed: from 0.644444 to 0.751111;	Best Classification Accuracy 0.751111， Best Classification loss 0.141309； Best Epoch： 4
epoch 5, train loss 0.155733, train acc 0.733, valida  loss 0.127491, valida acc 0.747
epoch 6, 

epoch 1, train loss 0.511815, train acc 0.244, valida  loss 0.384653, valida acc 0.391
Best_Val_Value changed: from 0.000000 to 0.391111;	Best Classification Accuracy 0.391111， Best Classification loss 0.384653； Best Epoch： 1
epoch 2, train loss 0.318278, train acc 0.396, valida  loss 0.224433, valida acc 0.536
Best_Val_Value changed: from 0.391111 to 0.535556;	Best Classification Accuracy 0.535556， Best Classification loss 0.224433； Best Epoch： 2
epoch 3, train loss 0.214974, train acc 0.587, valida  loss 0.188590, valida acc 0.647
Best_Val_Value changed: from 0.535556 to 0.646667;	Best Classification Accuracy 0.646667， Best Classification loss 0.188590； Best Epoch： 3
epoch 4, train loss 0.192192, train acc 0.676, valida  loss 0.128780, valida acc 0.671
Best_Val_Value changed: from 0.646667 to 0.671111;	Best Classification Accuracy 0.671111， Best Classification loss 0.128780； Best Epoch： 4
epoch 5, train loss 0.141616, train acc 0.751, valida  loss 0.111068, valida acc 0.773
Best_Val_

epoch 67, train loss 0.141254, train acc 0.953, valida  loss 0.023323, valida acc 0.940
Best_Val_Value changed: from 0.928889 to 0.940000;	Best Classification Accuracy 0.940000， Best Classification loss 0.023323； Best Epoch： 67
epoch 68, train loss 0.137052, train acc 0.924, valida  loss 0.050730, valida acc 0.871
Counter 1 of 20
epoch 69, train loss 0.157022, train acc 0.804, valida  loss 0.093588, valida acc 0.729
Counter 2 of 20
epoch 70, train loss 0.065155, train acc 0.842, valida  loss 0.077135, valida acc 0.836
Counter 3 of 20
epoch 71, train loss 0.089088, train acc 0.911, valida  loss 0.061431, valida acc 0.853
Counter 4 of 20
epoch 72, train loss 0.043388, train acc 0.909, valida  loss 0.059442, valida acc 0.833
Counter 5 of 20
epoch 73, train loss 0.106599, train acc 0.900, valida  loss 0.040885, valida acc 0.851
Counter 6 of 20
epoch 74, train loss 0.095810, train acc 0.900, valida  loss 0.033278, valida acc 0.907
Counter 7 of 20
epoch 75, train loss 0.079979, train acc 0.9

epoch 25, train loss 0.121254, train acc 0.900, valida  loss 0.071205, valida acc 0.862
epoch 26, train loss 0.088562, train acc 0.887, valida  loss 0.058344, valida acc 0.873
epoch 27, train loss 0.074357, train acc 0.913, valida  loss 0.061459, valida acc 0.842
epoch 28, train loss 0.060961, train acc 0.853, valida  loss 0.067416, valida acc 0.842
epoch 29, train loss 0.086114, train acc 0.900, valida  loss 0.052119, valida acc 0.887
epoch 30, train loss 0.042931, train acc 0.933, valida  loss 0.048649, valida acc 0.882
epoch 31, train loss 0.084029, train acc 0.947, valida  loss 0.037174, valida acc 0.931
Best_Val_Value changed: from 0.902222 to 0.931111;	Best Classification Accuracy 0.931111， Best Classification loss 0.037174； Best Epoch： 31
epoch 32, train loss 0.106733, train acc 0.873, valida  loss 0.051868, valida acc 0.873
epoch 33, train loss 0.080529, train acc 0.898, valida  loss 0.055146, valida acc 0.847
epoch 34, train loss 0.166556, train acc 0.864, valida  loss 0.05008

In [10]:
sys.path.append("./am3net/")

from model_am3net import *
from tool_am3net import *


windowSize = 25
BATCH_SIZE = 256
EPOCH = 100
FileName = 'am3net'
numComponents = 15

data_hsi, _ = applyPCA(data_hsi_o, numComponents=numComponents)
data_lidar, _ = applyPCA(data_lidar, numComponents=1)



train_dataset = Multidata(data_hsi, data_lidar, train_gt, windowSize)
train_loader = torch.utils.data.DataLoader(train_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)

val_dataset = Multidata(data_hsi, data_lidar, val_gt, windowSize)
val_loader = torch.utils.data.DataLoader(val_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)

test_dataset = Multidata(data_hsi, data_lidar, test_gt, windowSize)
test_loader = torch.utils.data.DataLoader(test_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)



KAPPA = []
OA = []
AA = []
ELEMENT_ACC = np.zeros((itm, num_classes))
training_time = []
testing_time = []


for itera in range(0, itm):
    
#     print("---------------------------------- Model Summary ---------------------------------------------")
    model = octfusion_multi_adder_1(in_channels_1=data_hsi.shape[2], in_channels_2=data_lidar.shape[2],
                                    out_channels=num_classes).to(DEVICE)
    summary(model,[(numComponents, windowSize,windowSize),(data_lidar.shape[2], windowSize,windowSize)]) 

    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=1)
    loss_func = MarginLoss(size_average=False, loss_lambda=0.25)
    
    model, train_time = train_am3net(model, loss_func, DEVICE, train_loader, optimizer, scheduler, EPOCH, val_loader, itera)
    test_acc_temp, test_loss_temp, y_pred, target, test_time = test(model, DEVICE, test_loader)
    oa,aa,kappa,each_acc, accuracy_matrix = reports(y_pred, target)
    
    input_1 = torch.randn(1, data_hsi.shape[2], windowSize, windowSize)
    input_2 = torch.randn(1, data_lidar.shape[2], windowSize, windowSize)
    input_1 = input_1.cuda()
    input_2 = input_2.cuda()
    macs, params = profile(model, inputs=(input_1,input_2, ))
    print("params, macs", params, macs)
    
    training_time.append(train_time)
    testing_time.append(test_time)
    KAPPA.append(kappa)
    OA.append(oa)
    AA.append(aa)
    ELEMENT_ACC[itera, :] = each_acc
    
    record.record_output(OA, AA, KAPPA, ELEMENT_ACC, training_time, testing_time, macs, params,train_sample, './' + FileName + '/' +
                         datasetName  + '_' + str(train_size) + '_' + str(params) + '_Report' +'.txt')

    print("final test results :", accuracy_matrix)

    del model,input_1,input_2


D:\anaconda3\envs\pytorch_envs\lib\site-packages\cupy\cuda\compiler.py:467: UserWarning: cupy.cuda.compile_with_cache has been deprecated in CuPy v10, and will be removed in the future. Use cupy.RawModule or cupy.RawKernel instead.
  ' instead.', UserWarning)
./am3net\model_am3net.py:126: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  weight_alpha = F.softmax(self.Weight_Alpha)


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 15, 25, 25]             240
            Conv2d-2           [-1, 16, 25, 25]             240
            Conv2d-3           [-1, 32, 25, 25]             512
            Conv2d-4           [-1, 64, 25, 25]           2,048
       Feature_HSI-5           [-1, 64, 25, 25]               0
            Conv2d-6            [-1, 3, 25, 25]              45
       BatchNorm2d-7            [-1, 3, 25, 25]               6
              ReLU-8            [-1, 3, 25, 25]               0
        ConvModule-9            [-1, 3, 25, 25]               0
           Conv2d-10           [-1, 45, 25, 25]             180
             ReLU-11           [-1, 45, 25, 25]               0
       ConvModule-12           [-1, 45, 25, 25]               0
       involution-13           [-1, 15, 25, 25]               0
           Conv2d-14           [-1, 64,

epoch 19, train loss 23.940532, train acc 0.871, valida  loss 49.006559, valida acc 0.733
epoch 20, train loss 22.586664, train acc 0.884, valida  loss 38.795120, valida acc 0.802
Best_Val_Value changed: from 0.757778 to 0.802222;	Best Classification Accuracy 0.802222， Best Classification loss 38.795120； Best Epoch： 20
epoch 21, train loss 19.933119, train acc 0.896, valida  loss 32.772120, valida acc 0.829
Best_Val_Value changed: from 0.802222 to 0.828889;	Best Classification Accuracy 0.828889， Best Classification loss 32.772120； Best Epoch： 21
epoch 22, train loss 20.397659, train acc 0.898, valida  loss 31.690172, valida acc 0.831
Best_Val_Value changed: from 0.828889 to 0.831111;	Best Classification Accuracy 0.831111， Best Classification loss 31.690172； Best Epoch： 22
epoch 23, train loss 17.821651, train acc 0.909, valida  loss 40.197405, valida acc 0.769
epoch 24, train loss 19.000453, train acc 0.904, valida  loss 38.309376, valida acc 0.784
epoch 25, train loss 17.784393, train

epoch 81, train loss 10.357667, train acc 0.947, valida  loss 28.231956, valida acc 0.844
Counter 9 of 20
epoch 82, train loss 10.685143, train acc 0.947, valida  loss 27.207627, valida acc 0.851
Counter 10 of 20
epoch 83, train loss 10.611927, train acc 0.944, valida  loss 27.395731, valida acc 0.851
Counter 11 of 20
epoch 84, train loss 10.603762, train acc 0.944, valida  loss 28.412029, valida acc 0.840
Counter 12 of 20
epoch 85, train loss 10.458501, train acc 0.947, valida  loss 27.117693, valida acc 0.853
Counter 13 of 20
epoch 86, train loss 10.523632, train acc 0.944, valida  loss 25.704785, valida acc 0.858
Counter 14 of 20
epoch 87, train loss 10.505204, train acc 0.947, valida  loss 25.409055, valida acc 0.869
Counter 15 of 20
epoch 88, train loss 10.298868, train acc 0.944, valida  loss 26.844909, valida acc 0.864
Counter 16 of 20
epoch 89, train loss 10.388151, train acc 0.947, valida  loss 27.973595, valida acc 0.851
Counter 17 of 20
epoch 90, train loss 10.308059, train 

D:\anaconda3\envs\pytorch_envs\lib\site-packages\sklearn\metrics\_classification.py:1308: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\anaconda3\envs\pytorch_envs\lib\site-packages\sklearn\metrics\_classification.py:1308: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\anaconda3\envs\pytorch_envs\lib\site-packages\sklearn\metrics\_classification.py:1308: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\anaconda3\envs\pytorch_envs\lib\si

...

epoch 47, train loss 13.465793, train acc 0.933, valida  loss 26.334566, valida acc 0.856
epoch 48, train loss 13.366580, train acc 0.933, valida  loss 26.405869, valida acc 0.856
epoch 49, train loss 13.339062, train acc 0.933, valida  loss 26.103187, valida acc 0.860
epoch 50, train loss 13.005711, train acc 0.933, valida  loss 26.052662, valida acc 0.862
epoch 51, train loss 13.337018, train acc 0.931, valida  loss 25.810595, valida acc 0.862
Counter 1 of 20
epoch 52, train loss 12.620769, train acc 0.936, valida  loss 26.166123, valida acc 0.853
Counter 2 of 20
epoch 53, train loss 12.790380, train acc 0.936, valida  loss 27.443328, valida acc 0.856
Counter 3 of 20
epoch 54, train loss 13.039868, train acc 0.936, valida  loss 27.282907, valida acc 0.853
Counter 4 of 20
epoch 55, train loss 12.702286, train acc 0.936, valida  loss 27.462572, valida acc 0.860
Counter 5 of 20
epoch 56, train loss 12.769195, train acc 0.936, valida  loss 28.724181, valida acc 0.842
Counter 6 of 20
epoc

D:\anaconda3\envs\pytorch_envs\lib\site-packages\sklearn\metrics\_classification.py:1308: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\anaconda3\envs\pytorch_envs\lib\site-packages\sklearn\metrics\_classification.py:1308: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\anaconda3\envs\pytorch_envs\lib\site-packages\sklearn\metrics\_classification.py:1308: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
./am3net\model_am3net.py:126: UserWar

epoch 1, train loss 156.036285, train acc 0.067, valida  loss 153.863098, valida acc 0.093
Best_Val_Value changed: from 0.000000 to 0.093333;	Best Classification Accuracy 0.093333， Best Classification loss 153.863098； Best Epoch： 1
epoch 2, train loss 150.057259, train acc 0.142, valida  loss 145.158440, valida acc 0.173
Best_Val_Value changed: from 0.093333 to 0.173333;	Best Classification Accuracy 0.173333， Best Classification loss 145.158440； Best Epoch： 2
epoch 3, train loss 137.233574, train acc 0.238, valida  loss 131.572491, valida acc 0.267
Best_Val_Value changed: from 0.173333 to 0.266667;	Best Classification Accuracy 0.266667， Best Classification loss 131.572491； Best Epoch： 3
epoch 4, train loss 122.407230, train acc 0.389, valida  loss 112.818939, valida acc 0.389
Best_Val_Value changed: from 0.266667 to 0.388889;	Best Classification Accuracy 0.388889， Best Classification loss 112.818939； Best Epoch： 4
epoch 5, train loss 105.421413, train acc 0.487, valida  loss 94.251949,

epoch 48, train loss 14.191384, train acc 0.931, valida  loss 26.519207, valida acc 0.860
epoch 49, train loss 14.063951, train acc 0.931, valida  loss 25.831066, valida acc 0.858
epoch 50, train loss 14.840953, train acc 0.929, valida  loss 25.202230, valida acc 0.864
epoch 51, train loss 14.649221, train acc 0.929, valida  loss 25.837399, valida acc 0.858
Counter 1 of 20
epoch 52, train loss 14.087540, train acc 0.931, valida  loss 25.959124, valida acc 0.858
Counter 2 of 20
epoch 53, train loss 13.619789, train acc 0.931, valida  loss 25.895110, valida acc 0.864
Counter 3 of 20
epoch 54, train loss 13.407604, train acc 0.933, valida  loss 26.630290, valida acc 0.856
Counter 4 of 20
epoch 55, train loss 13.407363, train acc 0.933, valida  loss 26.696840, valida acc 0.860
Counter 5 of 20
epoch 56, train loss 13.235788, train acc 0.933, valida  loss 25.846770, valida acc 0.862
Counter 6 of 20
epoch 57, train loss 13.135862, train acc 0.933, valida  loss 25.137223, valida acc 0.860
Coun

D:\anaconda3\envs\pytorch_envs\lib\site-packages\sklearn\metrics\_classification.py:1308: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\anaconda3\envs\pytorch_envs\lib\site-packages\sklearn\metrics\_classification.py:1308: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\anaconda3\envs\pytorch_envs\lib\site-packages\sklearn\metrics\_classification.py:1308: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
./am3net\model_am3net.py:126: UserWar

In [18]:
sys.path.append("./crossHL/")
from CrossHL import *


windowSize = 11
BATCH_SIZE = 64
EPOCH = 200
FM = 16
num_heads = 8
mlp_dim = 512
depth = 2
# ntokens = 4
FileName = 'crossHL'

data = data_hsi_o.reshape(np.prod(data_hsi_o.shape[:2]), np.prod(data_hsi_o.shape[2:]))
scaler = preprocessing.MinMaxScaler()
data = scaler.fit_transform(data)
data_hsi = data.reshape(data_hsi_o.shape[0], data_hsi_o.shape[1], data_hsi_o.shape[2])
data_lidar = data_lidar_o


# data_hsi, _ = applyPCA(data_hsi_o, numComponents=numComponents)
# data_lidar, _ = applyPCA(data_lidar, numComponents=1)

train_dataset = Multidata(data_hsi, data_lidar, train_gt, windowSize)
train_loader = torch.utils.data.DataLoader(train_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)

val_dataset = Multidata(data_hsi, data_lidar, val_gt, windowSize)
val_loader = torch.utils.data.DataLoader(val_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)

test_dataset = Multidata(data_hsi, data_lidar, test_gt, windowSize)
test_loader = torch.utils.data.DataLoader(test_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)



KAPPA = []
OA = []
AA = []
ELEMENT_ACC = np.zeros((itm, num_classes))
training_time = []
testing_time = []


for itera in range(0, itm):
    
#     print("---------------------------------- Model Summary ---------------------------------------------")

    model = CrossHL_Transformer(FM=FM, NC=hsi_band, NCLidar=lidar_band, Classes=num_classes,patchsize = windowSize).cuda()
    
    summary(model,[(hsi_band, windowSize,windowSize),(lidar_band, windowSize,windowSize)]) 
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4,weight_decay=5e-3)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.9)
    loss_func = nn.CrossEntropyLoss()
    
    model, train_time = train(model, loss_func, DEVICE, train_loader, optimizer, scheduler, EPOCH, val_loader, itera)
    test_acc_temp, test_loss_temp, y_pred, target, test_time = test(model, DEVICE, test_loader)
    oa,aa,kappa,each_acc, accuracy_matrix = reports(y_pred, target)
    
    input_1 = torch.randn(1, hsi_band, windowSize, windowSize)
    input_2 = torch.randn(1, lidar_band, windowSize, windowSize)
    input_1 = input_1.cuda()
    input_2 = input_2.cuda()
    macs, params = profile(model, inputs=(input_1,input_2, ))
    print("params, macs", params, macs)
    
    training_time.append(train_time)
    testing_time.append(test_time)
    KAPPA.append(kappa)
    OA.append(oa)
    AA.append(aa)
    ELEMENT_ACC[itera, :] = each_acc
    
    record.record_output(OA, AA, KAPPA, ELEMENT_ACC, training_time, testing_time, macs, params,train_sample, './' + FileName + '/' +
                         datasetName  + '_' + str(train_size) + '_' + str(params) + '_Report' +'.txt')

    print("final test results :", accuracy_matrix)

    del model,input_1,input_2


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv3d-1       [-1, 8, 136, 11, 11]             656
       BatchNorm3d-2       [-1, 8, 136, 11, 11]              16
              ReLU-3       [-1, 8, 136, 11, 11]               0
            Conv2d-4           [-1, 64, 11, 11]          39,232
            Conv2d-5           [-1, 64, 11, 11]          69,696
           HetConv-6           [-1, 64, 11, 11]               0
       BatchNorm2d-7           [-1, 64, 11, 11]             128
              ReLU-8           [-1, 64, 11, 11]               0
           Dropout-9              [-1, 122, 64]               0
        LayerNorm-10              [-1, 122, 64]             128
           Linear-11               [-1, 1, 512]          61,952
           Linear-12              [-1, 122, 64]           4,096
           Linear-13               [-1, 64, 64]           7,808
           Linear-14              [-1, 

epoch 35, train loss 0.937956, train acc 0.751, valida  loss 1.003457, valida acc 0.753
epoch 36, train loss 0.613514, train acc 0.800, valida  loss 0.951430, valida acc 0.669
epoch 37, train loss 0.613574, train acc 0.827, valida  loss 0.787756, valida acc 0.767
epoch 38, train loss 0.499982, train acc 0.816, valida  loss 0.668655, valida acc 0.756
epoch 39, train loss 0.494700, train acc 0.840, valida  loss 0.810788, valida acc 0.827
Best_Val_Value changed: from 0.817778 to 0.826667;	Best Classification Accuracy 0.826667， Best Classification loss 0.810788； Best Epoch： 39
epoch 40, train loss 0.569644, train acc 0.860, valida  loss 0.881111, valida acc 0.831
Best_Val_Value changed: from 0.826667 to 0.831111;	Best Classification Accuracy 0.831111， Best Classification loss 0.881111； Best Epoch： 40
epoch 41, train loss 0.506130, train acc 0.836, valida  loss 1.179254, valida acc 0.593
epoch 42, train loss 0.511558, train acc 0.842, valida  loss 0.518088, valida acc 0.871
Best_Val_Value c

epoch 116, train loss 0.236885, train acc 0.933, valida  loss 0.358951, valida acc 0.900
Counter 16 of 20
epoch 117, train loss 0.277585, train acc 0.947, valida  loss 0.629388, valida acc 0.891
Counter 17 of 20
epoch 118, train loss 0.312556, train acc 0.927, valida  loss 0.568908, valida acc 0.884
Counter 18 of 20
epoch 119, train loss 0.246668, train acc 0.920, valida  loss 0.391744, valida acc 0.887
Counter 19 of 20
epoch 120, train loss 0.348704, train acc 0.924, valida  loss 0.373081, valida acc 0.900
Counter 20 of 20
epoch 121, train loss 0.794365, train acc 0.936, valida  loss 0.463886, valida acc 0.869
Counter 21 of 20
Early stopping with best_val_acc:  0.9044444444444445 at epoch 95: ...
||======= Train Time for 0:03:00.959727 ======||

Test set: Average loss: 0.0000, Accuracy: 12513/14129 (88.5625%)
||======= Test Time for 0:00:14.600651 ======||
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv3d'>.
[INFO] Register count_bn() for <class 'torch.nn.modules

epoch 11, train loss 1.095852, train acc 0.653, valida  loss 0.948305, valida acc 0.658
Best_Val_Value changed: from 0.657778 to 0.657778;	Best Classification Accuracy 0.657778， Best Classification loss 0.948305； Best Epoch： 11
epoch 12, train loss 1.072091, train acc 0.704, valida  loss 0.939737, valida acc 0.658
Best_Val_Value changed: from 0.657778 to 0.657778;	Best Classification Accuracy 0.657778， Best Classification loss 0.939737； Best Epoch： 12
epoch 13, train loss 1.115800, train acc 0.691, valida  loss 1.147950, valida acc 0.602
epoch 14, train loss 1.167491, train acc 0.693, valida  loss 0.939514, valida acc 0.680
Best_Val_Value changed: from 0.657778 to 0.680000;	Best Classification Accuracy 0.680000， Best Classification loss 0.939514； Best Epoch： 14
epoch 15, train loss 1.047058, train acc 0.693, valida  loss 0.921926, valida acc 0.682
Best_Val_Value changed: from 0.680000 to 0.682222;	Best Classification Accuracy 0.682222， Best Classification loss 0.921926； Best Epoch： 15


epoch 81, train loss 0.574610, train acc 0.820, valida  loss 0.607322, valida acc 0.782
epoch 82, train loss 0.461987, train acc 0.851, valida  loss 0.630356, valida acc 0.833
epoch 83, train loss 0.513946, train acc 0.862, valida  loss 0.532584, valida acc 0.816
epoch 84, train loss 0.654632, train acc 0.862, valida  loss 0.682941, valida acc 0.816
epoch 85, train loss 0.454552, train acc 0.858, valida  loss 0.580416, valida acc 0.836
epoch 86, train loss 0.660485, train acc 0.867, valida  loss 0.491776, valida acc 0.844
epoch 87, train loss 0.463995, train acc 0.867, valida  loss 0.934821, valida acc 0.778
epoch 88, train loss 0.808488, train acc 0.829, valida  loss 0.512610, valida acc 0.836
epoch 89, train loss 0.741023, train acc 0.836, valida  loss 0.748592, valida acc 0.809
epoch 90, train loss 0.436078, train acc 0.889, valida  loss 0.587739, valida acc 0.847
epoch 91, train loss 0.378835, train acc 0.891, valida  loss 0.511213, valida acc 0.844
epoch 92, train loss 0.375312, t

...

epoch 1, train loss 2.601584, train acc 0.162, valida  loss 2.745796, valida acc 0.084
Best_Val_Value changed: from 0.000000 to 0.084444;	Best Classification Accuracy 0.084444， Best Classification loss 2.745796； Best Epoch： 1
epoch 2, train loss 2.342343, train acc 0.262, valida  loss 2.503340, valida acc 0.178
Best_Val_Value changed: from 0.084444 to 0.177778;	Best Classification Accuracy 0.177778， Best Classification loss 2.503340； Best Epoch： 2
epoch 3, train loss 1.860711, train acc 0.427, valida  loss 2.479457, valida acc 0.149
epoch 4, train loss 1.873770, train acc 0.469, valida  loss 2.423695, valida acc 0.229
Best_Val_Value changed: from 0.177778 to 0.228889;	Best Classification Accuracy 0.228889， Best Classification loss 2.423695； Best Epoch： 4
epoch 5, train loss 1.481132, train acc 0.496, valida  loss 2.055198, valida acc 0.242
Best_Val_Value changed: from 0.228889 to 0.242222;	Best Classification Accuracy 0.242222， Best Classification loss 2.055198； Best Epoch： 5
epoch 6, 

epoch 69, train loss 0.305484, train acc 0.893, valida  loss 0.444631, valida acc 0.858
epoch 70, train loss 0.393722, train acc 0.900, valida  loss 0.444877, valida acc 0.844
epoch 71, train loss 0.348623, train acc 0.900, valida  loss 0.601445, valida acc 0.807
epoch 72, train loss 0.289700, train acc 0.909, valida  loss 0.406913, valida acc 0.867
Best_Val_Value changed: from 0.866667 to 0.866667;	Best Classification Accuracy 0.866667， Best Classification loss 0.406913； Best Epoch： 72
epoch 73, train loss 0.584292, train acc 0.907, valida  loss 0.421137, valida acc 0.878
Best_Val_Value changed: from 0.866667 to 0.877778;	Best Classification Accuracy 0.877778， Best Classification loss 0.421137； Best Epoch： 73
epoch 74, train loss 0.481821, train acc 0.862, valida  loss 0.803423, valida acc 0.784
epoch 75, train loss 0.424921, train acc 0.867, valida  loss 0.602025, valida acc 0.798
epoch 76, train loss 0.348600, train acc 0.880, valida  loss 0.462783, valida acc 0.862
epoch 77, train 

epoch 1, train loss 2.862926, train acc 0.078, valida  loss 2.650456, valida acc 0.120
Best_Val_Value changed: from 0.000000 to 0.120000;	Best Classification Accuracy 0.120000， Best Classification loss 2.650456； Best Epoch： 1
epoch 2, train loss 2.388664, train acc 0.238, valida  loss 2.726185, valida acc 0.096
epoch 3, train loss 2.045821, train acc 0.287, valida  loss 2.793341, valida acc 0.136
Best_Val_Value changed: from 0.120000 to 0.135556;	Best Classification Accuracy 0.135556， Best Classification loss 2.793341； Best Epoch： 3
epoch 4, train loss 1.840781, train acc 0.380, valida  loss 2.700814, valida acc 0.178
Best_Val_Value changed: from 0.135556 to 0.177778;	Best Classification Accuracy 0.177778， Best Classification loss 2.700814； Best Epoch： 4
epoch 5, train loss 1.602734, train acc 0.416, valida  loss 2.596405, valida acc 0.207
Best_Val_Value changed: from 0.177778 to 0.206667;	Best Classification Accuracy 0.206667， Best Classification loss 2.596405； Best Epoch： 5
epoch 6, 

epoch 61, train loss 0.379257, train acc 0.856, valida  loss 0.545747, valida acc 0.813
epoch 62, train loss 0.518442, train acc 0.887, valida  loss 0.568134, valida acc 0.804
epoch 63, train loss 0.326395, train acc 0.878, valida  loss 1.358351, valida acc 0.769
epoch 64, train loss 0.760571, train acc 0.862, valida  loss 0.648879, valida acc 0.836
epoch 65, train loss 0.463007, train acc 0.889, valida  loss 0.501483, valida acc 0.827
epoch 66, train loss 0.408183, train acc 0.889, valida  loss 0.823746, valida acc 0.756
epoch 67, train loss 0.372171, train acc 0.862, valida  loss 0.746382, valida acc 0.778
epoch 68, train loss 0.652733, train acc 0.889, valida  loss 0.532305, valida acc 0.851
Best_Val_Value changed: from 0.851111 to 0.851111;	Best Classification Accuracy 0.851111， Best Classification loss 0.532305； Best Epoch： 68
epoch 69, train loss 0.411643, train acc 0.887, valida  loss 0.806957, valida acc 0.824
epoch 70, train loss 0.384228, train acc 0.876, valida  loss 0.77912

epoch 133, train loss 0.258528, train acc 0.900, valida  loss 0.539936, valida acc 0.807
Counter 1 of 20
epoch 134, train loss 0.517197, train acc 0.889, valida  loss 0.507637, valida acc 0.851
Counter 2 of 20
epoch 135, train loss 0.241453, train acc 0.924, valida  loss 0.472468, valida acc 0.833
Counter 3 of 20
epoch 136, train loss 0.252157, train acc 0.922, valida  loss 0.482916, valida acc 0.867
Counter 4 of 20
epoch 137, train loss 0.222725, train acc 0.936, valida  loss 0.356465, valida acc 0.889
Counter 5 of 20
epoch 138, train loss 0.321847, train acc 0.927, valida  loss 0.498751, valida acc 0.887
Counter 6 of 20
epoch 139, train loss 0.220342, train acc 0.940, valida  loss 0.408657, valida acc 0.862
Counter 7 of 20
epoch 140, train loss 0.301443, train acc 0.931, valida  loss 0.414248, valida acc 0.862
Counter 8 of 20
epoch 141, train loss 0.219189, train acc 0.936, valida  loss 0.550508, valida acc 0.827
Counter 9 of 20
epoch 142, train loss 0.227850, train acc 0.947, valida

In [19]:
sys.path.append("./S2ENet/")
from S2ENet import *

windowSize = 7
BATCH_SIZE = 64
EPOCH = 128
FileName = 'S2ENet'

data = data_hsi_o.reshape(np.prod(data_hsi_o.shape[:2]), np.prod(data_hsi_o.shape[2:]))
scaler = preprocessing.MinMaxScaler()
data = scaler.fit_transform(data)
data_hsi = data.reshape(data_hsi_o.shape[0], data_hsi_o.shape[1], data_hsi_o.shape[2])

data_l = data_lidar_o.reshape(np.prod(data_lidar_o.shape[:2]), np.prod(data_lidar_o.shape[2:]))
scaler = preprocessing.MinMaxScaler()
data_l = scaler.fit_transform(data_l)
data_lidar = data_l.reshape(data_lidar_o.shape[0], data_lidar_o.shape[1], data_lidar_o.shape[2])


train_dataset = Multidata(data_hsi, data_lidar, train_gt, windowSize)
train_loader = torch.utils.data.DataLoader(train_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)

val_dataset = Multidata(data_hsi, data_lidar, val_gt, windowSize)
val_loader = torch.utils.data.DataLoader(val_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)

test_dataset = Multidata(data_hsi, data_lidar, test_gt, windowSize)
test_loader = torch.utils.data.DataLoader(test_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)



KAPPA = []
OA = []
AA = []
ELEMENT_ACC = np.zeros((itm, num_classes))
training_time = []
testing_time = []


for itera in range(0, itm):
    
#     print("---------------------------------- Model Summary ---------------------------------------------")

    model = S2ENet(hsi_band, lidar_band, num_classes, patch_size = windowSize).cuda()  
    summary(model,[(hsi_band, windowSize,windowSize),(lidar_band, windowSize,windowSize)]) 
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=1)
    loss_func = nn.CrossEntropyLoss()
    
    model, train_time = train(model, loss_func, DEVICE, train_loader, optimizer, scheduler, EPOCH, val_loader, itera)
    test_acc_temp, test_loss_temp, y_pred, target, test_time = test(model, DEVICE, test_loader)
    oa,aa,kappa,each_acc, accuracy_matrix = reports(y_pred, target)
    
    input_1 = torch.randn(1, hsi_band, windowSize, windowSize)
    input_2 = torch.randn(1, lidar_band, windowSize, windowSize)
    input_1 = input_1.cuda()
    input_2 = input_2.cuda()
    macs, params = profile(model, inputs=(input_1,input_2, ))
    print("params, macs", params, macs)
    
    training_time.append(train_time)
    testing_time.append(test_time)
    KAPPA.append(kappa)
    OA.append(oa)
    AA.append(aa)
    ELEMENT_ACC[itera, :] = each_acc
    
    record.record_output(OA, AA, KAPPA, ELEMENT_ACC, training_time, testing_time, macs, params,train_sample, './' + FileName + '/' +
                         datasetName  + '_' + str(train_size) + '_' + str(params) + '_Report' +'.txt')

    print("final test results :", accuracy_matrix)

    del model,input_1,input_2


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1            [-1, 128, 7, 7]         166,016
       BatchNorm2d-2            [-1, 128, 7, 7]             256
              ReLU-3            [-1, 128, 7, 7]               0
      conv_bn_relu-4            [-1, 128, 7, 7]               0
            Conv2d-5              [-1, 8, 7, 7]              80
       BatchNorm2d-6              [-1, 8, 7, 7]              16
              ReLU-7              [-1, 8, 7, 7]               0
      conv_bn_relu-8              [-1, 8, 7, 7]               0
            Conv2d-9             [-1, 64, 7, 7]          73,792
      BatchNorm2d-10             [-1, 64, 7, 7]             128
             ReLU-11             [-1, 64, 7, 7]               0
     conv_bn_relu-12             [-1, 64, 7, 7]               0
           Conv2d-13             [-1, 16, 7, 7]           1,168
      BatchNorm2d-14             [-1, 1

epoch 30, train loss 1.017321, train acc 0.818, valida  loss 1.081702, valida acc 0.662
epoch 31, train loss 0.948005, train acc 0.840, valida  loss 0.865191, valida acc 0.822
Best_Val_Value changed: from 0.775556 to 0.822222;	Best Classification Accuracy 0.822222， Best Classification loss 0.865191； Best Epoch： 31
epoch 32, train loss 0.948323, train acc 0.831, valida  loss 0.798570, valida acc 0.836
Best_Val_Value changed: from 0.822222 to 0.835556;	Best Classification Accuracy 0.835556， Best Classification loss 0.798570； Best Epoch： 32
epoch 33, train loss 0.925892, train acc 0.869, valida  loss 0.727888, valida acc 0.842
Best_Val_Value changed: from 0.835556 to 0.842222;	Best Classification Accuracy 0.842222， Best Classification loss 0.727888； Best Epoch： 33
epoch 34, train loss 0.759671, train acc 0.856, valida  loss 0.811261, valida acc 0.811
epoch 35, train loss 0.906595, train acc 0.842, valida  loss 0.743912, valida acc 0.816
epoch 36, train loss 0.839582, train acc 0.864, vali

epoch 111, train loss 0.509856, train acc 0.927, valida  loss 0.393096, valida acc 0.900
Counter 7 of 20
epoch 112, train loss 0.427137, train acc 0.942, valida  loss 0.372256, valida acc 0.884
Counter 8 of 20
epoch 113, train loss 0.384508, train acc 0.947, valida  loss 0.550707, valida acc 0.847
Counter 9 of 20
epoch 114, train loss 0.372383, train acc 0.940, valida  loss 0.525175, valida acc 0.816
Counter 10 of 20
epoch 115, train loss 0.332842, train acc 0.958, valida  loss 0.429832, valida acc 0.858
Counter 11 of 20
epoch 116, train loss 0.484134, train acc 0.942, valida  loss 0.414621, valida acc 0.876
Counter 12 of 20
epoch 117, train loss 0.553035, train acc 0.947, valida  loss 0.412283, valida acc 0.867
Counter 13 of 20
epoch 118, train loss 0.428064, train acc 0.953, valida  loss 0.274773, valida acc 0.918
Best_Val_Value changed: from 0.917778 to 0.917778;	Best Classification Accuracy 0.917778， Best Classification loss 0.274773； Best Epoch： 118
epoch 119, train loss 0.323899,

epoch 6, train loss 2.067673, train acc 0.416, valida  loss 2.095197, valida acc 0.402
Best_Val_Value changed: from 0.326667 to 0.402222;	Best Classification Accuracy 0.402222， Best Classification loss 2.095197； Best Epoch： 6
epoch 7, train loss 1.967912, train acc 0.449, valida  loss 1.997290, valida acc 0.371
epoch 8, train loss 1.899210, train acc 0.436, valida  loss 1.898568, valida acc 0.442
Best_Val_Value changed: from 0.402222 to 0.442222;	Best Classification Accuracy 0.442222， Best Classification loss 1.898568； Best Epoch： 8
epoch 9, train loss 1.816675, train acc 0.529, valida  loss 1.936571, valida acc 0.493
Best_Val_Value changed: from 0.442222 to 0.493333;	Best Classification Accuracy 0.493333， Best Classification loss 1.936571； Best Epoch： 9
epoch 10, train loss 1.845944, train acc 0.547, valida  loss 1.770427, valida acc 0.516
Best_Val_Value changed: from 0.493333 to 0.515556;	Best Classification Accuracy 0.515556， Best Classification loss 1.770427； Best Epoch： 10
epoch 1

epoch 76, train loss 0.409478, train acc 0.918, valida  loss 0.537507, valida acc 0.840
epoch 77, train loss 0.634449, train acc 0.938, valida  loss 0.722427, valida acc 0.747
epoch 78, train loss 0.611482, train acc 0.958, valida  loss 0.485847, valida acc 0.842
epoch 79, train loss 0.463181, train acc 0.936, valida  loss 0.444901, valida acc 0.867
epoch 80, train loss 0.509050, train acc 0.933, valida  loss 0.477156, valida acc 0.827
epoch 81, train loss 0.435030, train acc 0.938, valida  loss 0.560879, valida acc 0.811
epoch 82, train loss 0.654208, train acc 0.940, valida  loss 0.716900, valida acc 0.798
epoch 83, train loss 0.411748, train acc 0.953, valida  loss 0.500945, valida acc 0.867
epoch 84, train loss 0.416808, train acc 0.951, valida  loss 0.347944, valida acc 0.887
Best_Val_Value changed: from 0.873333 to 0.886667;	Best Classification Accuracy 0.886667， Best Classification loss 0.347944； Best Epoch： 84
epoch 85, train loss 0.493884, train acc 0.949, valida  loss 0.64829

...

epoch 69, train loss 0.587617, train acc 0.918, valida  loss 0.498043, valida acc 0.871
Best_Val_Value changed: from 0.868889 to 0.871111;	Best Classification Accuracy 0.871111， Best Classification loss 0.498043； Best Epoch： 69
epoch 70, train loss 0.711519, train acc 0.911, valida  loss 1.232716, valida acc 0.589
epoch 71, train loss 0.617957, train acc 0.902, valida  loss 0.757069, valida acc 0.771
epoch 72, train loss 0.690028, train acc 0.918, valida  loss 0.737310, valida acc 0.733
epoch 73, train loss 0.584524, train acc 0.918, valida  loss 0.765143, valida acc 0.802
epoch 74, train loss 0.661463, train acc 0.909, valida  loss 1.012712, valida acc 0.647
epoch 75, train loss 0.469038, train acc 0.911, valida  loss 0.555540, valida acc 0.838
epoch 76, train loss 0.529293, train acc 0.913, valida  loss 0.531959, valida acc 0.840
epoch 77, train loss 0.534150, train acc 0.922, valida  loss 0.505287, valida acc 0.833
epoch 78, train loss 0.497985, train acc 0.931, valida  loss 0.88051

epoch 1, train loss 2.631013, train acc 0.178, valida  loss 2.677893, valida acc 0.122
Best_Val_Value changed: from 0.000000 to 0.122222;	Best Classification Accuracy 0.122222， Best Classification loss 2.677893； Best Epoch： 1
epoch 2, train loss 2.417828, train acc 0.378, valida  loss 2.652786, valida acc 0.236
Best_Val_Value changed: from 0.122222 to 0.235556;	Best Classification Accuracy 0.235556， Best Classification loss 2.652786； Best Epoch： 2
epoch 3, train loss 2.256160, train acc 0.416, valida  loss 2.534566, valida acc 0.311
Best_Val_Value changed: from 0.235556 to 0.311111;	Best Classification Accuracy 0.311111， Best Classification loss 2.534566； Best Epoch： 3
epoch 4, train loss 2.165090, train acc 0.444, valida  loss 2.189667, valida acc 0.431
Best_Val_Value changed: from 0.311111 to 0.431111;	Best Classification Accuracy 0.431111， Best Classification loss 2.189667； Best Epoch： 4
epoch 5, train loss 2.073549, train acc 0.536, valida  loss 2.218229, valida acc 0.367
epoch 6, 

epoch 58, train loss 0.720287, train acc 0.904, valida  loss 0.830549, valida acc 0.722
epoch 59, train loss 0.798692, train acc 0.882, valida  loss 0.760935, valida acc 0.802
epoch 60, train loss 0.630180, train acc 0.913, valida  loss 0.846176, valida acc 0.693
epoch 61, train loss 0.759114, train acc 0.842, valida  loss 0.755655, valida acc 0.824
epoch 62, train loss 0.576490, train acc 0.911, valida  loss 0.619456, valida acc 0.818
epoch 63, train loss 0.817429, train acc 0.922, valida  loss 0.580740, valida acc 0.838
epoch 64, train loss 0.677660, train acc 0.907, valida  loss 0.745195, valida acc 0.736
epoch 65, train loss 0.551924, train acc 0.909, valida  loss 0.686655, valida acc 0.782
epoch 66, train loss 0.690633, train acc 0.911, valida  loss 0.973681, valida acc 0.824
epoch 67, train loss 0.663351, train acc 0.913, valida  loss 0.796443, valida acc 0.793
epoch 68, train loss 0.569697, train acc 0.909, valida  loss 0.583221, valida acc 0.836
epoch 69, train loss 0.473110, t

epoch 1, train loss 2.603621, train acc 0.156, valida  loss 2.681132, valida acc 0.091
Best_Val_Value changed: from 0.000000 to 0.091111;	Best Classification Accuracy 0.091111， Best Classification loss 2.681132； Best Epoch： 1
epoch 2, train loss 2.394377, train acc 0.302, valida  loss 2.513178, valida acc 0.111
Best_Val_Value changed: from 0.091111 to 0.111111;	Best Classification Accuracy 0.111111， Best Classification loss 2.513178； Best Epoch： 2
epoch 3, train loss 2.332012, train acc 0.418, valida  loss 2.381116, valida acc 0.347
Best_Val_Value changed: from 0.111111 to 0.346667;	Best Classification Accuracy 0.346667， Best Classification loss 2.381116； Best Epoch： 3
epoch 4, train loss 2.151109, train acc 0.504, valida  loss 2.199806, valida acc 0.469
Best_Val_Value changed: from 0.346667 to 0.468889;	Best Classification Accuracy 0.468889， Best Classification loss 2.199806； Best Epoch： 4
epoch 5, train loss 2.078321, train acc 0.520, valida  loss 2.026010, valida acc 0.522
Best_Val_

epoch 60, train loss 0.719461, train acc 0.916, valida  loss 0.974884, valida acc 0.738
epoch 61, train loss 0.665423, train acc 0.924, valida  loss 0.780628, valida acc 0.804
epoch 62, train loss 0.522752, train acc 0.931, valida  loss 0.558174, valida acc 0.862
epoch 63, train loss 0.454630, train acc 0.947, valida  loss 0.897078, valida acc 0.718
epoch 64, train loss 0.653903, train acc 0.904, valida  loss 0.912590, valida acc 0.802
epoch 65, train loss 0.599540, train acc 0.944, valida  loss 0.849407, valida acc 0.860
epoch 66, train loss 0.466419, train acc 0.918, valida  loss 0.589987, valida acc 0.847
epoch 67, train loss 0.613212, train acc 0.922, valida  loss 0.486299, valida acc 0.873
Best_Val_Value changed: from 0.864444 to 0.873333;	Best Classification Accuracy 0.873333， Best Classification loss 0.486299； Best Epoch： 67
epoch 68, train loss 0.443467, train acc 0.936, valida  loss 0.468890, valida acc 0.858
epoch 69, train loss 0.682637, train acc 0.931, valida  loss 0.45577

In [25]:
sys.path.append("./CFCNN/")
from tool_cfcnn import *

windowSize = 11
BATCH_SIZE = 64
EPOCH = 150
FileName = 'CFCNN'

data = data_hsi_o.reshape(np.prod(data_hsi_o.shape[:2]), np.prod(data_hsi_o.shape[2:]))
scaler = preprocessing.MinMaxScaler()
data = scaler.fit_transform(data)
data_hsi = data.reshape(data_hsi_o.shape[0], data_hsi_o.shape[1], data_hsi_o.shape[2])

data_l = data_lidar_o.reshape(np.prod(data_lidar_o.shape[:2]), np.prod(data_lidar_o.shape[2:]))
scaler = preprocessing.MinMaxScaler()
data_l = scaler.fit_transform(data_l)
data_lidar = data_l.reshape(data_lidar_o.shape[0], data_lidar_o.shape[1], data_lidar_o.shape[2])


train_dataset = Multidata(data_hsi, data_lidar, train_gt, windowSize)
train_loader = torch.utils.data.DataLoader(train_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)

val_dataset = Multidata(data_hsi, data_lidar, val_gt, windowSize)
val_loader = torch.utils.data.DataLoader(val_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)

test_dataset = Multidata(data_hsi, data_lidar, test_gt, windowSize)
test_loader = torch.utils.data.DataLoader(test_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=True)



KAPPA = []
OA = []
AA = []
ELEMENT_ACC = np.zeros((itm, num_classes))
training_time = []
testing_time = []


for itera in range(0, itm):
    
#     print("---------------------------------- Model Summary ---------------------------------------------")

    model = Cross_fusion_CNN(hsi_band, lidar_band, num_classes).cuda()  
    summary(model,[(hsi_band, windowSize,windowSize),(lidar_band, windowSize,windowSize)]) 
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer=optimizer, gamma=0.96)
    loss_func = nn.CrossEntropyLoss()
    
    model, train_time = traincfcnn(model, loss_func, DEVICE, train_loader, optimizer, scheduler, EPOCH, val_loader, itera)
    test_acc_temp, test_loss_temp, y_pred, target, test_time = testcfcnn(model, DEVICE, test_loader)
    oa,aa,kappa,each_acc, accuracy_matrix = reports(y_pred, target)
    
    input_1 = torch.randn(1, hsi_band, windowSize, windowSize)
    input_2 = torch.randn(1, lidar_band, windowSize, windowSize)
    input_1 = input_1.cuda()
    input_2 = input_2.cuda()
    macs, params = profile(model, inputs=(input_1,input_2, ))
    print("params, macs", params, macs)
    
    training_time.append(train_time)
    testing_time.append(test_time)
    KAPPA.append(kappa)
    OA.append(oa)
    AA.append(aa)
    ELEMENT_ACC[itera, :] = each_acc
    
    record.record_output(OA, AA, KAPPA, ELEMENT_ACC, training_time, testing_time, macs, params,train_sample, './' + FileName + '/' +
                         datasetName  + '_' + str(train_size) + '_' + str(params) + '_Report' +'.txt')

    print("final test results :", accuracy_matrix)

    del model,input_1,input_2


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 16, 11, 11]          20,752
       BatchNorm2d-2           [-1, 16, 11, 11]              32
              ReLU-3           [-1, 16, 11, 11]               0
            Conv2d-4           [-1, 32, 11, 11]             544
       BatchNorm2d-5           [-1, 32, 11, 11]              64
              ReLU-6           [-1, 32, 11, 11]               0
         MaxPool2d-7             [-1, 32, 6, 6]               0
            Conv2d-8             [-1, 64, 6, 6]          18,496
       BatchNorm2d-9             [-1, 64, 6, 6]             128
             ReLU-10             [-1, 64, 6, 6]               0
           Conv2d-11           [-1, 16, 11, 11]             160
      BatchNorm2d-12           [-1, 16, 11, 11]              32
             ReLU-13           [-1, 16, 11, 11]               0
           Conv2d-14           [-1, 32,

epoch 24, train loss 1.452320, train acc 0.933, valida  loss 3.839995, valida acc 0.762
Best_Val_Value changed: from 0.753333 to 0.762222;	Best Classification Accuracy 0.762222， Best Classification loss 3.839995； Best Epoch： 24
epoch 25, train loss 1.362984, train acc 0.947, valida  loss 4.094717, valida acc 0.744
epoch 26, train loss 1.296957, train acc 0.940, valida  loss 3.831554, valida acc 0.733
epoch 27, train loss 1.397732, train acc 0.933, valida  loss 3.695401, valida acc 0.758
epoch 28, train loss 1.338872, train acc 0.944, valida  loss 3.792763, valida acc 0.758
epoch 29, train loss 1.359759, train acc 0.951, valida  loss 3.968798, valida acc 0.747
epoch 30, train loss 1.344424, train acc 0.938, valida  loss 3.844649, valida acc 0.771
Best_Val_Value changed: from 0.762222 to 0.771111;	Best Classification Accuracy 0.771111， Best Classification loss 3.844649； Best Epoch： 30
epoch 31, train loss 1.308933, train acc 0.962, valida  loss 3.857164, valida acc 0.758
epoch 32, train 

epoch 110, train loss 1.309849, train acc 0.960, valida  loss 4.440801, valida acc 0.709
Counter 10 of 20
epoch 111, train loss 1.257029, train acc 0.958, valida  loss 3.929428, valida acc 0.744
Counter 11 of 20
epoch 112, train loss 1.308465, train acc 0.956, valida  loss 4.168483, valida acc 0.729
Counter 12 of 20
epoch 113, train loss 1.354886, train acc 0.947, valida  loss 4.159192, valida acc 0.753
Counter 13 of 20
epoch 114, train loss 1.323698, train acc 0.951, valida  loss 4.185878, valida acc 0.724
Counter 14 of 20
epoch 115, train loss 1.379350, train acc 0.942, valida  loss 4.264359, valida acc 0.740
Counter 15 of 20
epoch 116, train loss 1.261287, train acc 0.956, valida  loss 4.257667, valida acc 0.722
Counter 16 of 20
epoch 117, train loss 1.288755, train acc 0.953, valida  loss 4.467634, valida acc 0.729
Counter 17 of 20
epoch 118, train loss 1.323411, train acc 0.956, valida  loss 4.651083, valida acc 0.718
Counter 18 of 20
epoch 119, train loss 1.302190, train acc 0.95

epoch 1, train loss 3.631996, train acc 0.042, valida  loss 4.075671, valida acc 0.022
Best_Val_Value changed: from 0.000000 to 0.022222;	Best Classification Accuracy 0.022222， Best Classification loss 4.075671； Best Epoch： 1
epoch 2, train loss 2.607993, train acc 0.193, valida  loss 3.710047, valida acc 0.031
Best_Val_Value changed: from 0.022222 to 0.031111;	Best Classification Accuracy 0.031111， Best Classification loss 3.710047； Best Epoch： 2
epoch 3, train loss 2.276502, train acc 0.480, valida  loss 3.598129, valida acc 0.211
Best_Val_Value changed: from 0.031111 to 0.211111;	Best Classification Accuracy 0.211111， Best Classification loss 3.598129； Best Epoch： 3
epoch 4, train loss 2.093603, train acc 0.687, valida  loss 3.776865, valida acc 0.293
Best_Val_Value changed: from 0.211111 to 0.293333;	Best Classification Accuracy 0.293333， Best Classification loss 3.776865； Best Epoch： 4
epoch 5, train loss 1.890133, train acc 0.771, valida  loss 3.520974, valida acc 0.389
Best_Val_

epoch 63, train loss 1.301414, train acc 0.953, valida  loss 2.775704, valida acc 0.744
epoch 64, train loss 1.336008, train acc 0.960, valida  loss 2.836371, valida acc 0.758
epoch 65, train loss 1.284394, train acc 0.964, valida  loss 2.839231, valida acc 0.751
epoch 66, train loss 1.308150, train acc 0.964, valida  loss 2.813821, valida acc 0.762
epoch 67, train loss 1.302538, train acc 0.958, valida  loss 2.805159, valida acc 0.789
epoch 68, train loss 1.385725, train acc 0.964, valida  loss 2.861923, valida acc 0.747
epoch 69, train loss 1.362208, train acc 0.960, valida  loss 2.875373, valida acc 0.753
epoch 70, train loss 1.312183, train acc 0.953, valida  loss 2.880547, valida acc 0.758
epoch 71, train loss 1.387416, train acc 0.967, valida  loss 2.809213, valida acc 0.753
epoch 72, train loss 1.312809, train acc 0.956, valida  loss 2.701692, valida acc 0.771
epoch 73, train loss 1.401311, train acc 0.949, valida  loss 2.698602, valida acc 0.758
epoch 74, train loss 1.357165, t

...

epoch 63, train loss 1.364319, train acc 0.942, valida  loss 3.191699, valida acc 0.778
epoch 64, train loss 1.472062, train acc 0.949, valida  loss 3.367388, valida acc 0.776
epoch 65, train loss 1.285669, train acc 0.962, valida  loss 3.299648, valida acc 0.762
epoch 66, train loss 1.441004, train acc 0.956, valida  loss 3.436767, valida acc 0.742
epoch 67, train loss 1.395488, train acc 0.942, valida  loss 3.147878, valida acc 0.780
epoch 68, train loss 1.318670, train acc 0.940, valida  loss 3.177165, valida acc 0.802
Best_Val_Value changed: from 0.795556 to 0.802222;	Best Classification Accuracy 0.802222， Best Classification loss 3.177165； Best Epoch： 68
epoch 69, train loss 1.341145, train acc 0.951, valida  loss 3.317950, valida acc 0.780
epoch 70, train loss 1.387495, train acc 0.944, valida  loss 3.176038, valida acc 0.784
epoch 71, train loss 1.399312, train acc 0.951, valida  loss 3.248430, valida acc 0.791
epoch 72, train loss 1.359425, train acc 0.947, valida  loss 3.26067

epoch 1, train loss 3.311304, train acc 0.191, valida  loss 4.541643, valida acc 0.091
Best_Val_Value changed: from 0.000000 to 0.091111;	Best Classification Accuracy 0.091111， Best Classification loss 4.541643； Best Epoch： 1
epoch 2, train loss 2.523124, train acc 0.438, valida  loss 4.679374, valida acc 0.091
Best_Val_Value changed: from 0.091111 to 0.091111;	Best Classification Accuracy 0.091111， Best Classification loss 4.679374； Best Epoch： 2
epoch 3, train loss 2.401367, train acc 0.547, valida  loss 4.224997, valida acc 0.207
Best_Val_Value changed: from 0.091111 to 0.206667;	Best Classification Accuracy 0.206667， Best Classification loss 4.224997； Best Epoch： 3
epoch 4, train loss 2.075975, train acc 0.622, valida  loss 4.042533, valida acc 0.313
Best_Val_Value changed: from 0.206667 to 0.313333;	Best Classification Accuracy 0.313333， Best Classification loss 4.042533； Best Epoch： 4
epoch 5, train loss 1.935434, train acc 0.707, valida  loss 3.981361, valida acc 0.431
Best_Val_

epoch 69, train loss 1.371921, train acc 0.949, valida  loss 2.815879, valida acc 0.802
epoch 70, train loss 1.305944, train acc 0.956, valida  loss 3.001762, valida acc 0.800
epoch 71, train loss 1.235926, train acc 0.947, valida  loss 2.716390, valida acc 0.816
epoch 72, train loss 1.248723, train acc 0.956, valida  loss 2.728052, valida acc 0.820
epoch 73, train loss 1.313126, train acc 0.942, valida  loss 3.092302, valida acc 0.822
epoch 74, train loss 1.321464, train acc 0.958, valida  loss 2.960513, valida acc 0.789
epoch 75, train loss 1.242175, train acc 0.956, valida  loss 2.947939, valida acc 0.796
epoch 76, train loss 1.322963, train acc 0.958, valida  loss 2.904883, valida acc 0.804
epoch 77, train loss 1.361711, train acc 0.949, valida  loss 2.934589, valida acc 0.804
epoch 78, train loss 1.365228, train acc 0.942, valida  loss 2.831922, valida acc 0.809
epoch 79, train loss 1.220644, train acc 0.956, valida  loss 3.173377, valida acc 0.807
epoch 80, train loss 1.370535, t

epoch 1, train loss 3.343346, train acc 0.176, valida  loss 7.134695, valida acc 0.062
Best_Val_Value changed: from 0.000000 to 0.062222;	Best Classification Accuracy 0.062222， Best Classification loss 7.134695； Best Epoch： 1
epoch 2, train loss 2.445472, train acc 0.360, valida  loss 5.483599, valida acc 0.084
Best_Val_Value changed: from 0.062222 to 0.084444;	Best Classification Accuracy 0.084444， Best Classification loss 5.483599； Best Epoch： 2
epoch 3, train loss 2.104953, train acc 0.542, valida  loss 4.242798, valida acc 0.189
Best_Val_Value changed: from 0.084444 to 0.188889;	Best Classification Accuracy 0.188889， Best Classification loss 4.242798； Best Epoch： 3
epoch 4, train loss 2.069068, train acc 0.691, valida  loss 3.924676, valida acc 0.227
Best_Val_Value changed: from 0.188889 to 0.226667;	Best Classification Accuracy 0.226667， Best Classification loss 3.924676； Best Epoch： 4
epoch 5, train loss 1.946116, train acc 0.722, valida  loss 3.643507, valida acc 0.342
Best_Val_

epoch 73, train loss 1.295335, train acc 0.953, valida  loss 2.847638, valida acc 0.820
epoch 74, train loss 1.260238, train acc 0.962, valida  loss 2.763673, valida acc 0.833
epoch 75, train loss 1.376465, train acc 0.967, valida  loss 2.946008, valida acc 0.829
epoch 76, train loss 1.245816, train acc 0.962, valida  loss 2.882506, valida acc 0.831
epoch 77, train loss 1.319558, train acc 0.960, valida  loss 2.896944, valida acc 0.822
epoch 78, train loss 1.338184, train acc 0.956, valida  loss 2.763022, valida acc 0.831
epoch 79, train loss 1.327074, train acc 0.960, valida  loss 2.708909, valida acc 0.827
epoch 80, train loss 1.232619, train acc 0.971, valida  loss 2.747165, valida acc 0.820
epoch 81, train loss 1.257769, train acc 0.951, valida  loss 2.754644, valida acc 0.824
epoch 82, train loss 1.341705, train acc 0.958, valida  loss 2.720663, valida acc 0.827
epoch 83, train loss 1.335087, train acc 0.953, valida  loss 2.911807, valida acc 0.827
epoch 84, train loss 1.284319, t